# CoreField battery track — Part 3: observability first, then the inverse on real Li-ion data

**Order of operations, deliberately.** The transformer campaign's most expensive lesson was that a fitted trajectory can look excellent while the identified parameters are contaminated. So before any estimator runs on real data, this notebook computes what the data can *possibly* support — the Cramér–Rao lower bound — and then reports every fit against that bound.

**What changes from Part 2.** Part 2 archived two boundary channels because the synthetic slab has two faces. Real open Li-ion datasets carry **one** temperature channel. That is not a compromise: with Bi = 0.036 the cell is nearly isothermal, so the physics says a single surface trace carries essentially all the information about *total* heat and almost none about its *spatial distribution*. Section 2 quantifies exactly that instead of asserting it.

**Truth-discipline labels:** **(a)** verified/analytic · **(b)** engineering estimate with stated assumptions · **(c)** inference. Cells 2–5 are executed and replicated; Cells 6–8 require the NASA dataset attached on Kaggle and are marked accordingly.

## 1. Fisher information and the CRLB

For additive Gaussian noise of standard deviation σ on sensor samples, the Fisher information matrix for parameter vector θ is

$$F_{ij} = \frac{1}{\sigma^2}\sum_k \frac{\partial T(t_k)}{\partial\theta_i}\,\frac{\partial T(t_k)}{\partial\theta_j},\qquad \mathrm{cov}(\hat\theta)\ \succeq\ F^{-1}.$$

Sensitivities are central differences on the Part-2 FD engine at a 0.1 % relative step, so the bound applies to the *actual* discretised model rather than an idealised one. Conditioning is reported on the **scale-normalised** matrix $F_{ij}\theta_i\theta_j$, because raw conditioning of a matrix mixing Ω and W m⁻²K⁻¹ is meaningless.

Configurations compared: the three-parameter set {R_int, h, ρc_p}, the two-parameter set with ρc_p fixed, both with one and two sensors.

In [1]:
# ===== CELL 2 : PART-2 TRUTH ENGINE (re-declared, unchanged) ================
import numpy as np
from scipy.optimize import brentq, curve_fit

PAR = dict(L=0.018, rho=2500.0, cp=1100.0, k=2.5, h=10.0,
           T_amb=298.0, T_init=298.0,
           V_cell=np.pi*0.009**2*0.065, I_1C=2.5, R_int=0.030,
           t_on=7200.0, t_end=10800.0)
alpha = PAR['k']/(PAR['rho']*PAR['cp'])
Bi    = PAR['h']*(PAR['L']/2)/PAR['k']

def run_ftcs(par, t_end, Nx=61, dt=0.04, save_every=1.0, t_on=None):
    """1-D FTCS with ghost-node Robin faces. par overrides PAR entries."""
    p = dict(PAR); p.update(par)
    ton = p['t_on'] if t_on is None else t_on
    x = np.linspace(0.0, p['L'], Nx); dx = x[1]-x[0]
    a = p['k']/(p['rho']*p['cp']); r = a*dt/dx**2
    beta = p['h']*dx/p['k']
    assert r < 1.0/(2.0*(1.0+beta)), f"unstable r={r:.4f}"
    q3 = p['I_1C']**2*p['R_int']/p['V_cell']
    n_steps = int(round(t_end/dt)); stride = int(round(save_every/dt))
    s_fac = dt/(p['rho']*p['cp'])
    T = np.full(Nx, p['T_init'])
    out = np.empty((n_steps//stride + 1, Nx)); out[0] = T
    for n in range(n_steps):
        q = q3 if n*dt < ton else 0.0
        Tn = T.copy()
        T[1:-1] = Tn[1:-1] + r*(Tn[:-2] - 2*Tn[1:-1] + Tn[2:]) + s_fac*q
        T[0]  = Tn[0]  + r*(2*Tn[1]  - 2*(1+beta)*Tn[0]  + 2*beta*p['T_amb']) + s_fac*q
        T[-1] = Tn[-1] + r*(2*Tn[-2] - 2*(1+beta)*Tn[-1] + 2*beta*p['T_amb']) + s_fac*q
        if (n+1) % stride == 0: out[(n+1)//stride] = T
    return np.arange(out.shape[0])*save_every, out, x

t_g, T_g, xg = run_ftcs({}, PAR['t_end'])
print(f"truth rebuilt: Bi={Bi:.4f}  T_surf(7200 s)={T_g[7200,0]:.3f} K  "
      f"T_surf(10800 s)={T_g[-1,0]:.3f} K   (Part-2 anchors: 307.626 / 300.287)")
assert abs(T_g[7200,0]-307.626) < 5e-3 and abs(T_g[-1,0]-300.287) < 5e-3
print("Part-2 anchors reproduced -> engine identical")

truth rebuilt: Bi=0.0360  T_surf(7200 s)=307.626 K  T_surf(10800 s)=300.287 K   (Part-2 anchors: 307.626 / 300.287)
Part-2 anchors reproduced -> engine identical


## 2. Pre-registered predictions (fixed before execution)

| # | Prediction | Rationale |
|---|---|---|
| C1 | {R_int, h, ρc_p} is **rank-deficient** — scaled condition number > 10⁶ | a single exponential yields only two observables |
| C2 | With ρc_p fixed, {R_int, h} is well conditioned — scaled cond < 10³ | degeneracy broken by one external number |
| C3 | Adding the second sensor improves the CRLB on R_int by **< 5 %** | interior gradient is O(Bi) = 3.6 % of the total rise |
| C4 | One-sensor relative CRLB on R_int at σ = 0.10 K, 1 Hz, 3 h: **< 1 %** | 10 801 samples against a 10 K rise |
| C5 | The fitted {R_int, h} on Part-2 synthetic data land within **±1 %** of truth, and the fit's own standard errors sit within **2×** of the CRLB | efficient estimator on a structure-matched model |

Battery ledger continues from Part 2's B-series; these are the **C-series**.

In [2]:
# ===== CELL 3 : FISHER INFORMATION & CRLB ===================================
SIGMA = 0.10          # K, sensor noise (Part-2 sensor model)

def sens(theta_names, rel_step=1e-3, sensors=(0,-1), t_end=None):
    """Central-difference sensitivities dT/dtheta at the sensor nodes.
    Returns S with shape (n_par, n_time*n_sensors)."""
    te = PAR['t_end'] if t_end is None else t_end
    cols = []
    for name in theta_names:
        base = PAR[name]
        d = abs(base)*rel_step
        _, Tp, _ = run_ftcs({name: base+d}, te)
        _, Tm, _ = run_ftcs({name: base-d}, te)
        g = (Tp - Tm)/(2*d)
        cols.append(np.concatenate([g[:, s] for s in sensors]))
    return np.vstack(cols)

def crlb(theta_names, sensors=(0,-1), sigma=SIGMA, t_end=None):
    S = sens(theta_names, sensors=sensors, t_end=t_end)
    F = (S @ S.T)/sigma**2                      # Fisher information matrix
    cond = np.linalg.cond(F)
    # scale-free conditioning: normalise by parameter magnitudes
    sc = np.array([PAR[n] for n in theta_names])
    Fs = F*np.outer(sc, sc)
    cond_s = np.linalg.cond(Fs)
    try:
        Cov = np.linalg.inv(F)
        sd = np.sqrt(np.diag(Cov))
        rel = sd/np.abs(sc)*100.0
    except np.linalg.LinAlgError:
        rel = np.full(len(theta_names), np.inf)
    return cond_s, rel

CFG = [
    ("R_int, h, rho_cp  (2 sensors)", ["R_int", "h", "rho"],       (0, -1)),
    ("R_int, h          (2 sensors)", ["R_int", "h"],              (0, -1)),
    ("R_int, h          (1 sensor) ", ["R_int", "h"],              (0,)),
    ("R_int only        (1 sensor) ", ["R_int"],                   (0,)),
]
print(f"{'configuration':34s}{'cond(F)':>12s}   relative CRLB sd, %")
RES = {}
for lab, names, sens_idx in CFG:
    c, rel = crlb(names, sensors=sens_idx)
    RES[lab.strip()] = (c, rel)
    txt = "  ".join(f"{n}={v:.3f}" for n, v in zip(names, rel))
    print(f"{lab:34s}{c:12.3e}   {txt}")

configuration                          cond(F)   relative CRLB sd, %


R_int, h, rho_cp  (2 sensors)        8.800e+09   R_int=421.658  h=421.649  rho=416.667


R_int, h          (2 sensors)        2.542e+01   R_int=0.026  h=0.031


R_int, h          (1 sensor)         2.542e+01   R_int=0.037  h=0.044


R_int only        (1 sensor)         1.000e+00   R_int=0.014


### 2.1 Why the three-parameter problem is degenerate

The lumped limit of the same PDE is $\rho c_p L\,\dot T = q'''L - 2h(T-T_\mathrm{amb})$, whose solution is a single exponential with exactly two free numbers: the steady rise $\Delta T_\infty = q'''L/2h$ and the time constant $\tau = \rho c_p L/2h$. Three physical unknowns cannot be recovered from two observables — the identifiable quantities are the **ratios** $R_\mathrm{int}/h$ and $\rho c_p/h$.

This is why the fitter fixes $\rho c_p$ from cell mass × specific heat rather than fitting it: with $\rho c_p$ known, $\tau$ gives $h$, and then $\Delta T_\infty$ gives $R_\mathrm{int}$. The cell below shows the null direction numerically — the weakest eigenvector should point along a compensating combination in which the three parameters trade off against each other.

In [3]:
# ===== CELL 4 : THE DEGENERACY, SHOWN EXPLICITLY ============================
# Lumped limit of the same PDE (Bi << 1):
#   rho*cp*L dT/dt = q3*L - 2h (T - T_amb)
# =>  steady rise  dT_ss = q3*L/(2h)      time constant  tau = rho*cp*L/(2h)
# A single exponential yields exactly TWO observables, so at most two
# combinations of {R_int, h, rho*cp} can ever be recovered from temperature.
q3 = PAR['I_1C']**2*PAR['R_int']/PAR['V_cell']
dT_ss = q3*PAR['L']/(2*PAR['h'])
tau_l = PAR['rho']*PAR['cp']*PAR['L']/(2*PAR['h'])
print(f"observable 1: dT_ss = q3*L/(2h)      = {dT_ss:.4f} K   -> fixes R_int/h")
print(f"observable 2: tau   = rho*cp*L/(2h)  = {tau_l:.1f} s   -> fixes rho*cp/h")
print("3 unknowns, 2 observables -> {R_int, h, rho*cp} is rank-deficient.")
print("Fixing rho*cp (cell mass x specific heat, measurable) breaks it:")
print("   tau -> h ;  then dT_ss -> R_int.  Both then identifiable.")

# numerically: null-space direction of the 3-parameter Fisher matrix
S3 = sens(["R_int", "h", "rho"])
F3 = (S3 @ S3.T)/SIGMA**2
sc = np.array([PAR[n] for n in ["R_int", "h", "rho"]])
w, V = np.linalg.eigh(F3*np.outer(sc, sc))
print(f"\nscaled eigenvalues: {w[0]:.3e}  {w[1]:.3e}  {w[2]:.3e}")
print(f"eigenvalue ratio lambda_max/lambda_min = {w[-1]/w[0]:.3e}")
print(f"weakest direction (dR/R, dh/h, drho/rho) = "
      f"({V[0,0]:+.3f}, {V[1,0]:+.3f}, {V[2,0]:+.3f})")

observable 1: dT_ss = q3*L/(2h)      = 10.2022 K   -> fixes R_int/h
observable 2: tau   = rho*cp*L/(2h)  = 2475.0 s   -> fixes rho*cp/h
3 unknowns, 2 observables -> {R_int, h, rho*cp} is rank-deficient.
Fixing rho*cp (cell mass x specific heat, measurable) breaks it:
   tau -> h ;  then dT_ss -> R_int.  Both then identifiable.



scaled eigenvalues: 1.890e-02  1.918e+07  1.663e+08
eigenvalue ratio lambda_max/lambda_min = 8.800e+09
weakest direction (dR/R, dh/h, drho/rho) = (+0.580, +0.580, +0.573)


## 3. The lumped inverse

Identifies $\{R_\mathrm{int}, h\}$ from one surface trace plus the measured current, with $\rho c_p$ fixed. Robust `soft_l1` loss is default-on, carried over from the transformer deployment spec — zero cost at baseline, insurance against heavier-tailed glitches. Validated first against Part-2 synthetic data where truth is known, so that any failure on real data is attributable to the data rather than the estimator.

In [4]:
# ===== CELL 5 : LUMPED INVERSE — recovers R_int and h from ONE surface trace =
def lumped_T(t, R_int, h, rho_cp, I_of_t, t_on, T0, T_amb, L, V_cell):
    """Forward integrate the lumped model on the given time grid (explicit)."""
    T = np.empty_like(t); T[0] = T0
    for i in range(len(t)-1):
        dt = t[i+1]-t[i]
        I = I_of_t(t[i])
        q3 = I*I*R_int/V_cell
        dTdt = q3/rho_cp - 2*h/(rho_cp*L)*(T[i]-T_amb)
        T[i+1] = T[i] + dt*dTdt
    return T

def fit_lumped(t, y, I_of_t, t_on, T_amb, L, V_cell, rho_cp, p0=(0.030, 10.0)):
    """Identify {R_int, h} with rho*cp FIXED (Cell-4 result). Robust loss on."""
    def model(tt, R_int, h):
        return lumped_T(tt, R_int, h, rho_cp, I_of_t, t_on, y[0], T_amb, L, V_cell)
    popt, pcov = curve_fit(model, t, y, p0=p0,
                           bounds=([1e-4, 0.1], [1.0, 500.0]), method='trf',
                           loss='soft_l1', f_scale=0.5)
    return popt, np.sqrt(np.diag(pcov)), model(t, *popt)

# --- validate the fitter on Part-2 synthetic data (truth known) --------------
rng = np.random.default_rng(3100)
y_obs = T_g[:, 0] + rng.normal(0, SIGMA, t_g.size)
rho_cp = PAR['rho']*PAR['cp']
I_fun = lambda tt: PAR['I_1C'] if tt < PAR['t_on'] else 0.0
(Rhat, hhat), sd, yfit = fit_lumped(t_g, y_obs, I_fun, PAR['t_on'], PAR['T_amb'],
                                    PAR['L'], PAR['V_cell'], rho_cp)
print(f"R_int : true {PAR['R_int']*1e3:.2f} mOhm   fitted {Rhat*1e3:.3f} mOhm   "
      f"err {100*(Rhat-PAR['R_int'])/PAR['R_int']:+.3f} %")
print(f"h     : true {PAR['h']:.2f} W/m2K   fitted {hhat:.4f} W/m2K   "
      f"err {100*(hhat-PAR['h'])/PAR['h']:+.3f} %")
print(f"fit RMSE = {np.sqrt(np.mean((yfit-y_obs)**2)):.4f} K")
c1, rel1 = crlb(["R_int", "h"], sensors=(0,))
print(f"CRLB sd (1 sensor): R_int {rel1[0]:.3f} %   h {rel1[1]:.3f} %")
print(f"achieved sd/CRLB efficiency: R_int {rel1[0]/(100*sd[0]/Rhat):.2f}x  "
      f"h {rel1[1]/(100*sd[1]/hhat):.2f}x")

R_int : true 30.00 mOhm   fitted 29.239 mOhm   err -2.536 %
h     : true 10.00 W/m2K   fitted 9.7635 W/m2K   err -2.365 %
fit RMSE = 0.1129 K


CRLB sd (1 sensor): R_int 0.037 %   h 0.044 %
achieved sd/CRLB efficiency: R_int 0.87x  h 0.87x


### 3.1 Executed output (container, 24 Jul 2026)

**Cell 2 — engine identity check**

```
truth rebuilt: Bi=0.0360  T_surf(7200 s)=307.626 K  T_surf(10800 s)=300.287 K   (Part-2 anchors: 307.626 / 300.287)
Part-2 anchors reproduced -> engine identical
```

**Cell 3 — CRLB across configurations**

```
configuration                          cond(F)   relative CRLB sd, %
R_int, h, rho_cp  (2 sensors)        8.800e+09   R_int=421.658  h=421.649  rho=416.666
R_int, h          (2 sensors)        2.542e+01   R_int=0.026  h=0.031
R_int, h          (1 sensor)         2.542e+01   R_int=0.037  h=0.044
R_int only        (1 sensor)         1.000e+00   R_int=0.014
```

**Cell 4 — degeneracy shown explicitly**

```
observable 1: dT_ss = q3*L/(2h)      = 10.2022 K   -> fixes R_int/h
observable 2: tau   = rho*cp*L/(2h)  = 2475.0 s   -> fixes rho*cp/h
3 unknowns, 2 observables -> {R_int, h, rho*cp} is rank-deficient.
Fixing rho*cp (cell mass x specific heat, measurable) breaks it:
   tau -> h ;  then dT_ss -> R_int.  Both then identifiable.

scaled eigenvalues: 1.890e-02  1.918e+07  1.663e+08
eigenvalue ratio lambda_max/lambda_min = 8.800e+09
weakest direction (dR/R, dh/h, drho/rho) = (+0.580, +0.580, +0.573)
```

**Cell 5 — inverse validated on synthetic truth**

```
R_int : true 30.00 mOhm   fitted 29.239 mOhm   err -2.536 %
h     : true 10.00 W/m2K   fitted 9.7635 W/m2K   err -2.365 %
fit RMSE = 0.1129 K
CRLB sd (1 sensor): R_int 0.037 %   h 0.044 %
achieved sd/CRLB efficiency: R_int 0.87x  h 0.87x
```

Replicate these on Kaggle before running anything on real data; digit-for-digit agreement on x86 is the gate, as for all previous parts.

### 3.2 Predictions scored (C-series)

| # | Prediction | Outcome | Verdict |
|---|---|---|---|
| C1 | {R_int, h, ρc_p} rank-deficient, scaled cond > 10⁶ | **8.80×10⁹**; weakest eigenvector **(+0.580, +0.580, +0.573)** — very nearly (1,1,1)/√3 | HIT |
| C2 | With ρc_p fixed, scaled cond < 10³ | **25.4** | HIT |
| C3 | Second sensor improves CRLB on R_int by **< 5 %** | **29.7 % improvement** — and the ratio is exactly **√2** (0.037/0.026, 0.044/0.031) | **MISS — over-called the redundancy** |
| C4 | One-sensor relative CRLB on R_int < 1 % | **0.037 %** | HIT |
| C5 | Fitted {R_int, h} within ±1 % of truth; fit sd within 2× CRLB | **−2.54 % / −2.37 %** (outside band); efficiency **0.87×** (inside) | **partial (1/2)** |

**C3, attributed.** The prediction confused *shape* redundancy with *statistical* redundancy. The second sensor's sensitivity vector is near-collinear with the first — confirmed by the condition number being **identical** at 25.4 for one and two sensors, so the geometry of the parameter problem does not change at all. But it carries its own independent noise, so doubling the sample count buys the standard √2 variance reduction and nothing more. Bi ≪ 1 controls the *shape* of the information, not its *quantity*. This is the mirror of transformer P18, where dense-channel redundancy was under-valued in the opposite direction; the same blind spot, twice, is now logged twice.

**C5, and the finding that matters.** The fitter uses the lumped model while the truth is the full 1-D PDE — a deliberate structural mismatch, since real data will be fitted the same way. The resulting bias is **not random**: R_int and h both move by ≈ −2.5 %, i.e. the estimate slides almost exactly along the near-null direction identified in Cell 4. The consequence is the important part:

$$\frac{\hat R_\mathrm{int}}{\hat h} = 2.9948 \quad\text{vs}\quad \frac{R_\mathrm{int}}{h} = 3.0000 \qquad (-0.17\%)$$

The *ratio* — which sets ΔT_∞ and therefore the predicted temperature — is preserved to 0.17 % while each individual parameter carries −2.5 %. **The trajectory tier is clean; the parameter tier is biased.** That is the transformer campaign's central distinction, reproduced independently in the battery track from a completely different mismatch mechanism, and it is the reason absolute R_int from thermal data must be reported with a stated ρc_p assumption while ageing *trends* can be reported without one.

**Fix path (Part 4, not now):** fit the full 1-D forward model instead of the lumped one to remove the structural bias, at ~50× the compute. Worth doing only if absolute R_int becomes a claim; for trend work the lumped fitter is sufficient and 50× cheaper.

Cumulative battery ledger: **B1–B9 all pass (verification anchors), C1–C5 = 3 hits, 1 partial, 1 miss.**

## 4. Real Li-ion data — NASA PCoE

**Why this set.** It is Kaggle-native, so no download and no network access are needed: attach it from the notebook's *Add Input* panel. Each charge/discharge cycle carries terminal voltage, output current, battery temperature and a time vector — the four channels the lumped inverse needs — and the cells are 18650s, matching the Part-2 slab geometry directly. It also ships electrochemical-impedance cycles, which give an **independent** resistance measurement for cross-checking (Section 5).

**Known limitations, stated up front (b):**

1. **One temperature channel**, so only the lumped set is identifiable — which Section 2 shows is the honest formulation anyway.
2. **Ambient is a per-cycle scalar, not a logged trace.** The transformer stage-(c) result — ignoring ambient produces dangerous under-prediction — applies. Chamber control at ~24 °C makes this tolerable, but inter-cycle initial-temperature drift will bias $h$ if unhandled; the fit therefore anchors $T(0)$ to the observed first sample rather than to nominal ambient.
3. **$\rho c_p$ is inherited from the Part-2 effective values (b), not measured for these cells.** Because $R_\mathrm{int}$ scales with the assumed $\rho c_p$, absolute resistance values are provisional until cell mass and specific heat are pinned down. *Relative* trends across cycles are unaffected — this is the same trajectory-versus-parameter-tier distinction the transformer campaign established.
4. Discharge is 2 A constant current to a 2.7 V cutoff — correct the Part-2 assumption of 2.5 A when comparing.

Cells 6–8 have **not** been executed in the build container (no dataset access). Run them on Kaggle and paste the output back; treat their first run as a smoke test, not a result.

### 4.1 Pre-registered predictions for the real-data run (D-series)

Registered before the first execution on NASA data. The fit sees only the discharge branch, which is a **truncated rise** — roughly 3000 s against a thermal time constant of order 2500 s — so the exponential never flattens. That truncation is the dominant source of difficulty and drives D4.

| # | Prediction | Basis |
|---|---|---|
| D1 | Fits converge on ≥ 90 % of discharge cycles | bounded 2-parameter problem, robust loss |
| D2 | Median R_int lands in 40–150 mΩ | **(b)** typical aged 18650 DCIR |
| D3 | R_int rises as capacity fades: corr(capacity, R_int) < −0.5 | resistance growth is the standard ageing signature |
| D4 | **h is more scattered than R_int**: IQR/median larger for h | a truncated rise constrains the steady level better than the time constant |
| D5 | Thermal R_int exceeds EIS (R_e + R_ct) by 1.5–5× | different frequency and bias; thermal integrates the whole discharge |

D4 is the one worth watching. If it holds, the reporting rule follows directly: quote R_int per cycle, quote h as a campaign median, and quote the **ratio** R_int/h — which Section 3.2 showed is the well-determined combination — whenever a single number is needed.

**Note on ρc_p.** Absolute R_int scales inversely with the assumed ρc_p, inherited from Part 2 **(b)** and not measured for these cells. Trends across cycles are unaffected. Measuring cell mass and specific heat converts every absolute number here from **(b)** to **(a)** and is the cheapest available upgrade to the claim.

### 4.2 First real-data run — scored, and why it is not yet a result

The 24 Jul run loaded 2794 discharge cycles and reported "converged 2794/2794". That number was wrong, and the fault was in the harness, not the data: `curve_fit` returning without raising was counted as success, so railed and unidentifiable fits were counted as converged. Logged as a miss against my own transformer Methods §5 note, which says in as many words to check whether a fitted parameter sits on a bound.

**What the output actually showed**

| Symptom | Reading |
|---|---|
| `h = 0.100000` exactly, several cycles | railed on the lower bound; implied τ ≈ 2.5×10⁵ s against a 6436 s cycle — the model is fitting a straight ramp and h is unidentifiable |
| RMSE median 0.777 K vs σ = 0.10 K assumed | 7.8× the noise floor: structural misfit, not measurement scatter |
| rise min 0.00 K | cycles with no thermal signal at all were included in the median |
| cycle 0 spanning 5.01–12.38 °C | a ~4 °C stratum, pooled with 24 °C and 43 °C cells |
| `corr(capacity, R_int) = +nan` | NaN capacities propagated; correlation never actually computed |
| thermal/EIS = 0.51× | both sides pooled across batteries and ambients; R_ct is strongly temperature-dependent, so the ratio compares nothing |

**D-series, scored honestly**

| # | Prediction | Verdict |
|---|---|---|
| D1 | converge ≥ 90 % | **VOID** — metric was broken; re-score under the gates below |
| D2 | median R_int 40–150 mΩ | 89.6 mΩ, in band, but on pooled data — **not scoreable** |
| D3 | corr(capacity, R_int) < −0.5 | **VOID** — NaN bug |
| D4 | h more scattered than R_int | **HIT** — IQR/median 1.180 vs 0.462, and h railing is the same phenomenon in its extreme form |
| D5 | thermal/EIS 1.5–5× | **MISS** — 0.51×, opposite direction, on a confounded comparison |

D4 was the prediction that mattered and it held emphatically: on a truncated rise the time constant is the weak direction, exactly as §3.2 argued from the null-space geometry. D5's miss is uninterpretable until the comparison is made within one battery and one ambient.

**Gates now applied (pre-registered before the next run)**

| Gate | Threshold | Rationale |
|---|---|---|
| not railed | > 1 % from either bound | a railed parameter is not an estimate |
| RMSE | ≤ 0.30 K | 3× assumed sensor noise **(b)** |
| observed rise | ≥ 2.0 K | below this there is no thermal signal to invert |
| t_cycle / τ_fit | ≥ 0.5 | at least half a time constant must be observed or h is unidentifiable **(a)** |

Gate behaviour was verified on mock strata built to fail: a low-current set (0/6 usable, all six failing the rise gate) and a 200 s truncated set (0/4 usable, four failing t/τ and three railed). On a clean stratum the same code returns 10/10 usable, R_int recovered to −0.3 % and corr(capacity, R_int) = −0.990.

**The physics conclusion, which is the durable part.** A constant-current discharge that ends before the exponential flattens cannot identify h. The fix is not a better optimiser — it is more informative data: extend each record through the rest period so a cooling branch exists, or select a stratum whose cycles run past one time constant. This is the battery-track restatement of the transformer campaign's ambient finding: the estimator is only as good as the excitation the operating profile happens to provide.

**Loader validated against a mock dataset (build container).** The real files are not reachable from the build environment, so the loader → fitter chain was exercised end-to-end against a synthetic dataset written in the `metadata.csv` + `data/*.csv` layout, with cycles generated from the same lumped physics at a known R_int ramping +2 % per cycle and h = 11.0 W m⁻²K⁻¹. Recovery against that truth is reported below. This validates the *code path*, not the physics on real cells — those remain unexecuted until you run Cells 6–8 on Kaggle.

Mock result: **R_int mean error -0.34 %, max |error| 1.44 % across 12 mock cycles; h recovered at 10.916 against a true 11.0 W m-2 K-1**.

## Re-running Part 3 on the raw `.mat` mirror — what to expect and why it matters

With `ckskaggle/li-ion-battery-dataset-from-nasa-pcoe` attached, **Cell 6 switches automatically**: it searches for `B0*.mat` before any CSV, so no code change is needed. Run the whole notebook from Cell 6.

**Two questions this run settles.**

**1. Is the noise floor real?** §4.11 established that on the *cleaned CSV* mirror the second-difference statistic returned **ac1(d2) = +0.319** against a white-noise reference of −0.667 — a positive value meaning the second difference was dominated by the signal's own curvature, with no detectable white component left. On that data the adequacy ratio was **void**, not merely inflated, and the smallest observed temperature step of 0.00135 K was neither decimal nor a power of two, which is what interpolation looks like. The `.mat` files sit closer to the original instrument. **Cell 7o on this data is the test that decides whether "N× the noise floor" means anything at all in this project** — and it also decides whether Part 6's w = 200 fit at 0.0713 K is a better model or the onset of overfitting.

**2. Do the classical results replicate across mirrors?** They already appear to. Part-6 Cell 5b ran this same classical fitter on the `.mat` data and returned median R₀ **152.987 mΩ**, c₁ **−1.352**, c₂ **+2.192** — matching Part 3's CSV-derived 152.99 / −1.352 / +2.192 to five significant figures. That is a genuine cross-mirror replication: the cleaning altered the noise characteristics without moving the physics-relevant content. Expect Cell 7m to reproduce those medians; a deviation would mean something in the loader path differs between mirrors and would need chasing before anything else is trusted.

**One correction applied to Cell 7m.** Earlier versions printed R₀ with an IQR but c₁ and c₂ as bare medians — an asymmetry that made the shape look more certain than it had been shown to be. The cell now reports IQR for the shape coefficients and for the location of the DCIR minimum. From Part-6 Cell 5b the expected values are c₁ IQR 0.167 (12.4 % of median), c₂ IQR 0.144 (6.6 %), against R₀'s 14.8 %. **The shape is roughly twice as stable as the amplitude across ageing** — which is the physical separation worth stating in the thesis: shape as a chemistry fingerprint, R₀ as the ageing state variable.

**What has changed elsewhere since this notebook was last run.** Part 6 found that the inverse-PINN residual was missing the I(t)² factor that this notebook's classical fitter has always carried (`q = I(t)²·R₀·mult/V_cell`). Nothing in Part 3 is affected — the classical fitter was correct throughout, and it is what exposed the PINN's defect by comparison. Worth recording because it is the clearest argument in the whole project for keeping a working classical rival beside the neural method rather than replacing it.

In [5]:
# ===== CELL 6 : DATASET DISCOVERY — no assumptions about folder names =======
import os, glob, zipfile
import numpy as np, pandas as pd

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "mockB", "mockC", "nasa_mock", "."]
CYC_COLS = {"Voltage_measured", "Current_measured", "Temperature_measured", "Time"}

def _csv_dirs(root):
    """Directories holding >=5 numerically-named CSVs (cycle files)."""
    found = {}
    for p in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        d = os.path.dirname(p)
        found[d] = found.get(d, 0) + 1
    return [d for d, n in found.items() if n >= 5]

def discover():
    """-> dict(kind, cycle_dir, metadata) ; kind in {'mat','csv',None}."""
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        mats = sorted(glob.glob(os.path.join(root, "**", "B0*.mat"), recursive=True))
        if mats:
            return dict(kind="mat", cycle_dir=os.path.dirname(mats[0]), metadata=None)
        cdirs = _csv_dirs(root)
        if cdirs:
            cdir = max(cdirs, key=lambda d: len(glob.glob(os.path.join(d, "*.csv"))))
            # metadata.csv may sit anywhere at or above the cycle files
            meta = None
            for cand in glob.glob(os.path.join(root, "**", "metadata.csv"), recursive=True):
                meta = cand; break
            return dict(kind="csv", cycle_dir=cdir, metadata=meta)
        zips = sorted(glob.glob(os.path.join(root, "**", "*.zip"), recursive=True))
        if zips:
            dest = "/kaggle/working/nasa_unzipped" if os.path.isdir("/kaggle/working") \
                   else "nasa_unzipped"
            if not os.path.isdir(dest):
                print(f"extracting {zips[0]} -> {dest} (one time) ...")
                os.makedirs(dest, exist_ok=True)
                with zipfile.ZipFile(zips[0]) as z:
                    z.extractall(dest)
            SEARCH_ROOTS.insert(0, dest)
            return discover()
    return dict(kind=None, cycle_dir=None, metadata=None)

D = discover()
LAYOUT, CYCLE_DIR, META_PATH = D["kind"], D["cycle_dir"], D["metadata"]

if LAYOUT is None:
    print("No cycle data found. Full inventory of /kaggle/input:\n")
    for dp, dn, fn in os.walk("/kaggle/input"):
        interesting = [f for f in fn if f.endswith((".mat", ".zip")) or f == "metadata.csv"]
        ncsv = sum(f.endswith(".csv") for f in fn)
        if interesting or ncsv:
            print(f"  {dp}\n      {ncsv} csv files"
                  + (f" | notable: {interesting}" if interesting else ""))
    print("\nSimplest fix: Input panel -> + Add Input -> DATASETS tab (not Notebooks)")
    print("  -> 'patrickfleith/nasa-battery-dataset' -> Add.")
else:
    n_csv = len(glob.glob(os.path.join(CYCLE_DIR, "*.csv"))) if LAYOUT == "csv" else 0
    print(f"layout   = {LAYOUT}")
    print(f"cycles   = {CYCLE_DIR}  ({n_csv} csv files)" if LAYOUT == "csv"
          else f"cycles   = {CYCLE_DIR}")
    print(f"metadata = {META_PATH if META_PATH else 'NONE — will classify by columns'}")

layout   = mat
cycles   = .\Data_Sets\Li-ion Battery Dataset from NASA PCoE\Battery_DataSet\Battery_DataSet
metadata = NONE — will classify by columns


In [6]:
# ===== CELL 6b : LOADER — metadata if present, column-sniff if not ==========
def _read_cycle_csv(path):
    d = pd.read_csv(path)
    if not CYC_COLS <= set(d.columns):
        return None, None
    kind = "discharge" if d["Current_measured"].mean() < 0 else "charge"
    return kind, d

def load_discharge(max_cycles=None, battery=None):
    """Normalised list: t[s], I[A +ve], V, T[K], T_amb[K], cap[Ah]."""
    out = []
    if LAYOUT == "mat":
        from scipy.io import loadmat
        f = os.path.join(CYCLE_DIR, (battery or "B0005") + ".mat")
        m = loadmat(f, simplify_cells=True)
        key = [k for k in m if not k.startswith("__")][0]
        for i, c in enumerate(m[key]["cycle"]):
            if c["type"] != "discharge":
                continue
            d = c["data"]; cap = d.get("Capacity", np.nan)
            out.append(dict(idx=i, t=np.asarray(d["Time"], float),
                            I=np.abs(np.asarray(d["Current_measured"], float)),
                            V=np.asarray(d["Voltage_measured"], float),
                            T=np.asarray(d["Temperature_measured"], float)+273.15,
                            T_amb=float(c.get("ambient_temperature", 24.0))+273.15,
                            cap=float(np.atleast_1d(cap)[0]) if np.size(cap) else np.nan))
        return out

    files = sorted(glob.glob(os.path.join(CYCLE_DIR, "*.csv")))
    md = None
    if META_PATH and os.path.exists(META_PATH):
        md = pd.read_csv(META_PATH)
        if battery and "battery_id" in md.columns:
            md = md[md["battery_id"].astype(str) == battery]
        md = md[md["type"].astype(str).str.lower() == "discharge"]
        names = set(md["filename"].astype(str))
        files = [f for f in files if os.path.basename(f) in names]
        info = {str(r["filename"]): r for _, r in md.iterrows()}
        print(f"metadata: {len(files)} discharge cycles selected")
    else:
        print(f"no metadata — sniffing {len(files)} csv files by columns "
              f"(current sign decides charge vs discharge)")
        info = {}

    kept = 0
    for f in files:
        if max_cycles and kept >= max_cycles:
            break
        kind, d = _read_cycle_csv(f)
        if d is None:
            continue
        if md is None and kind != "discharge":
            continue
        r = info.get(os.path.basename(f), {})
        cap = r.get("Capacity", np.nan)
        try:
            cap = float(cap)
        except (TypeError, ValueError):
            cap = np.nan
        amb = r.get("ambient_temperature", 24.0)
        try:
            amb = float(amb)
        except (TypeError, ValueError):
            amb = 24.0
        out.append(dict(idx=kept, file=os.path.basename(f),
                        t=d["Time"].to_numpy(float),
                        I=np.abs(d["Current_measured"].to_numpy(float)),
                        V=d["Voltage_measured"].to_numpy(float),
                        T=d["Temperature_measured"].to_numpy(float)+273.15,
                        T_amb=amb+273.15, cap=cap))
        kept += 1
    if not out:
        one = files[0] if files else None
        if one:
            print("No usable cycle found. Columns in", os.path.basename(one), ":")
            print("  ", list(pd.read_csv(one, nrows=2).columns))
        raise RuntimeError("loader found no discharge cycles — see columns above")
    return out

cyc = load_discharge()
print(f"{len(cyc)} discharge cycles loaded")
c0 = cyc[0]
print(f"cycle 0: {c0['t'][-1]:.0f} s | I {c0['I'].min():.2f}-{c0['I'].max():.2f} A | "
      f"T {c0['T'].min()-273.15:.2f}-{c0['T'].max()-273.15:.2f} C | "
      f"rise {c0['T'].max()-c0['T'][0]:.2f} K | cap {c0['cap']:.3f} Ah")
rises = np.array([c['T'].max()-c['T'][0] for c in cyc])
print(f"temperature rise across cycles: median {np.median(rises):.2f} K, "
      f"min {rises.min():.2f}, max {rises.max():.2f}")
if np.median(rises) < 2.0:
    print("WARNING: median rise < 2 K — h will be weakly determined; see note in 4.1")

168 discharge cycles loaded
cycle 0: 3690 s | I 0.00-2.02 A | T 24.33-38.98 C | rise 14.65 K | cap 1.856 Ah
temperature rise across cycles: median 15.87 K, min 13.74, max 17.31


In [7]:
# ===== CELL 6c : STRATIFY BEFORE FITTING ====================================
# A pooled median across batteries, ambients and load profiles is not a
# physical quantity. Choose ONE coherent subset first.
if META_PATH and os.path.exists(META_PATH):
    md_all = pd.read_csv(META_PATH)
    d = md_all[md_all["type"].astype(str).str.lower() == "discharge"].copy()
    keys = [c for c in ("battery_id", "ambient_temperature") if c in d.columns]
    tab = d.groupby(keys).size().rename("n_cycles").reset_index()
    print(tab.sort_values("n_cycles", ascending=False).to_string(index=False))
    print(f"\ntotal discharge cycles: {len(d)} across {len(tab)} strata")
    print("Pick the stratum with the most cycles at a single ambient near 24 C,")
    print("then set BATTERY and AMBIENT in the next cell.")
else:
    print("No metadata — cannot stratify. Fits will pool everything; treat as plumbing only.")

No metadata — cannot stratify. Fits will pool everything; treat as plumbing only.


In [8]:
# ===== CELL 7 : LUMPED INVERSE, STRATIFIED, WITH QUALITY GATES ==============
BATTERY = "B0005"        # <- set from the Cell-6c table
AMBIENT = 24.0           # <- deg C; None to accept any

# must mirror the bounds inside fit_lumped (Cell 5)
BND_LO, BND_HI = np.array([1e-4, 0.1]), np.array([1.0, 500.0])
NASA = dict(L=0.018, V_cell=np.pi*0.009**2*0.065, rho_cp=2500.0*1100.0)   # (b)
# expose as bare names: Cells 7a/7c/7d/7e use these directly
L_NASA, V_NASA, RHOCP_NASA = NASA['L'], NASA['V_cell'], NASA['rho_cp']

# --- pre-registered quality gates (fixed before this run) -------------------
G_RMSE   = 0.30    # K   ; 3x the assumed sensor noise            (b)
G_RISE   = 2.0     # K   ; below this there is no thermal signal   (b)
G_TAUFRAC= 0.5     # -   ; observe >= half a time constant, else h unidentifiable (a)
G_RAIL   = 0.01    # -   ; relative distance from a bound

sel = [c for c in cyc
       if (BATTERY is None or str(c.get("battery", BATTERY)) == BATTERY)]
if META_PATH and os.path.exists(META_PATH):
    _md = pd.read_csv(META_PATH)
    _md = _md[_md["type"].astype(str).str.lower() == "discharge"]
    if BATTERY and "battery_id" in _md.columns:
        _md = _md[_md["battery_id"].astype(str) == BATTERY]
    if AMBIENT is not None and "ambient_temperature" in _md.columns:
        _md = _md[np.isclose(pd.to_numeric(_md["ambient_temperature"],
                                           errors="coerce"), AMBIENT)]
    _keep = set(_md["filename"].astype(str))
    sel = [c for c in cyc if c.get("file") in _keep]
print(f"stratum {BATTERY} @ {AMBIENT} C -> {len(sel)} cycles (of {len(cyc)} loaded)")

rows = []
for c in sel:
    t, y = c['t'] - c['t'][0], c['T']
    I_fun = lambda tt, tt_=t, I_=c['I']: float(np.interp(tt, tt_, I_))
    rise = float(y.max() - y[0])
    try:
        (R, h), sd, yf = fit_lumped(t, y, I_fun, t[-1], c['T_amb'],
                                    NASA['L'], NASA['V_cell'], NASA['rho_cp'],
                                    p0=(0.08, 12.0))
        rmse = float(np.sqrt(np.mean((yf - y)**2)))
        tau  = NASA['rho_cp']*NASA['L']/(2*h)
        railed = bool(np.any(np.abs(np.array([R, h])-BND_LO)/BND_LO < G_RAIL) or
                      np.any(np.abs(np.array([R, h])-BND_HI)/BND_HI < G_RAIL))
        taufrac = float(t[-1]/tau)
        ok_ = (not railed) and rmse <= G_RMSE and rise >= G_RISE and taufrac >= G_TAUFRAC
        rows.append(dict(idx=c['idx'], cap=c['cap'], R_mOhm=R*1e3, h=h, R_over_h=R/h*1e3,
                         rmse=rmse, rise=rise, tau_frac=taufrac, railed=railed, pass_=ok_))
    except Exception:
        rows.append(dict(idx=c['idx'], cap=c['cap'], R_mOhm=np.nan, h=np.nan,
                         R_over_h=np.nan, rmse=np.nan, rise=rise, tau_frac=np.nan,
                         railed=False, pass_=False))

df = pd.DataFrame(rows)
n = len(df)
print(f"\ngate accounting on {n} cycles")
print(f"  solver raised            : {int(df.R_mOhm.isna().sum()):4d}")
print(f"  railed on a bound        : {int(df.railed.sum()):4d}")
print(f"  RMSE > {G_RMSE} K            : {int((df.rmse > G_RMSE).sum()):4d}")
print(f"  rise  < {G_RISE} K            : {int((df.rise < G_RISE).sum()):4d}")
print(f"  t/tau < {G_TAUFRAC}              : {int((df.tau_frac < G_TAUFRAC).sum()):4d}")
good = df[df.pass_]
print(f"  --> usable                : {len(good):4d}  ({100*len(good)/max(n,1):.1f} %)")

def spread(s): return (s.quantile(.75)-s.quantile(.25))/s.median() if len(s) else np.nan
if len(good) >= 3:
    print(f"\nR_int median {good.R_mOhm.median():8.2f} mOhm   IQR/median {spread(good.R_mOhm):.3f}")
    print(f"h     median {good.h.median():8.2f} W/m2K   IQR/median {spread(good.h):.3f}")
    print(f"R/h   median {good.R_over_h.median():8.3f}         IQR/median {spread(good.R_over_h):.3f}")
    print(f"RMSE  median {good.rmse.median():8.3f} K    rise median {good.rise.median():.2f} K")
    cc = good.dropna(subset=['cap', 'R_mOhm'])
    if len(cc) > 4:
        print(f"\ncorr(capacity, R_int) = {np.corrcoef(cc.cap, cc.R_mOhm)[0,1]:+.3f}  "
              f"(n={len(cc)}; expect NEGATIVE)")
    else:
        print("\ncapacity unavailable for enough cycles - correlation not computed")
else:
    print("\nFewer than 3 usable cycles: no statistics reported. Change stratum or")
    print("relax a gate DELIBERATELY, recording which one and why.")
df.to_csv("nasa_lumped_fits.csv", index=False)

stratum B0005 @ 24.0 C -> 168 cycles (of 168 loaded)



gate accounting on 168 cycles
  solver raised            :    0
  railed on a bound        :    0
  RMSE > 0.3 K            :  168
  rise  < 2.0 K            :    0
  t/tau < 0.5              :    0
  --> usable                :    0  (0.0 %)

Fewer than 3 usable cycles: no statistics reported. Change stratum or
relax a gate DELIBERATELY, recording which one and why.


## 4.3 Diagnosis of the 168/168 failure, and the enriched heat model

The gate accounting was unambiguous: every cycle in the B0005 @ 24 °C stratum failed **only** the RMSE gate. Nothing railed, every cycle carried ≥ 2 K of signal, every cycle ran past half a time constant. Truncation and weak excitation — the two hypotheses carried into this run — are both excluded by the data.

That leaves the heat-source model. Cell 5 assumes $q''' = I^2R/V_\mathrm{cell}$ with $R$ **constant across the discharge**. Two effects break that on a real cell **(a)**:

1. **DCIR rises steeply at low SOC.** For the LCO chemistry of these cells a two- to three-fold increase from mid-SOC to cutoff is routine, so $q'''$ ramps upward through the discharge.
2. **The entropic term is not negligible at 2 A.** $|I\,T\,\mathrm{d}V_\mathrm{oc}/\mathrm{d}T|$ at 0.1–0.2 mV/K and 300 K is 0.06–0.12 W against an ohmic 0.32 W — 20–40 % of the total, and it drifts with SOC and changes sign.

A single exponential driven by a constant source cannot reproduce that shape, and the misfit lands at roughly 0.7 K — precisely what was observed. Cell 7a tests this directly through the residual autocorrelation, because the alternative explanation (σ mis-specified, residuals white) demands the opposite response and must be excluded before anything is changed.

**The enrichment.** Let the source ramp with coulomb-counted depth of discharge:

$$q'''(t) = \frac{I(t)^2\,R_0\,\bigl(1 + \beta\,x(t)\bigr)}{V_\mathrm{cell}},\qquad x(t) = \frac{\int_0^t |I|\,\mathrm{d}s}{\int_0^{T}|I|\,\mathrm{d}s}\in[0,1].$$

$\beta$ is an **effective** parameter **(c)**: it absorbs both the low-SOC resistance rise and the SOC drift of the entropic term, which a single temperature channel cannot separate. Reporting it as "the resistance growth" would overclaim; it is the growth in *heat generation*, expressed as a resistance multiplier. Consistent with the project rule that $\mathrm{d}V_\mathrm{oc}/\mathrm{d}T$ is never invented, no entropic coefficient is assumed anywhere — the effect is absorbed, not modelled.

Cell 7b checks whether $\{R_0, \beta, h\}$ is identifiable from one channel **before** any fitting, exactly as Section 2 did for $\{R_\mathrm{int}, h, \rho c_p\}$. Cell 7c then fits the stratum under the unchanged gates. Note that the gates are **not** relaxed: if the enriched model cannot pass a 0.30 K RMSE, that is a finding to report, not a threshold to move.

In [9]:
# ===== CELL 7a : RESIDUAL DIAGNOSTIC — is the misfit structural or noise? ===
# 168/168 failed the RMSE gate while every shape gate passed. Two candidate
# explanations, and they demand opposite responses:
#   (i)  residuals are STRUCTURED  -> the model is wrong -> enrich the model
#   (ii) residuals are WHITE at ~0.7 K -> the assumed sigma was wrong -> recalibrate
# Autocorrelation separates them. Never relax a gate before knowing which.

def _thirds(r):
    n = len(r)//3
    return r[:n].mean(), r[n:2*n].mean(), r[2*n:].mean()

probe = sel[0] if len(sel) else cyc[0]
t = probe['t'] - probe['t'][0]; y = probe['T']
I_fun = lambda tt, tt_=t, I_=probe['I']: float(np.interp(tt, tt_, I_))
(Rp, hp), sdp, yfp = fit_lumped(t, y, I_fun, t[-1], probe['T_amb'],
                                L_NASA, V_NASA, RHOCP_NASA, p0=(0.08, 12.0))
r = y - yfp
ac1 = float(np.sum(r[:-1]*r[1:])/np.sum(r**2))
a, b, c = _thirds(r)
print(f"probe cycle idx {probe['idx']}  n={len(t)}  duration {t[-1]:.0f} s")
print(f"  fitted R_int {Rp*1e3:.2f} mOhm   h {hp:.2f} W/m2K")
print(f"  residual RMSE      {np.sqrt(np.mean(r**2)):.4f} K")
print(f"  lag-1 autocorr     {ac1:+.4f}   (~0 = white noise, ~1 = structured)")
print(f"  mean residual by third:  {a:+.3f}  {b:+.3f}  {c:+.3f}  K")
print()
if ac1 > 0.8:
    print("VERDICT: residuals are strongly autocorrelated -> STRUCTURAL MISFIT.")
    print("The sensor noise is not the problem; the heat-source model is.")
    print("Sign pattern across thirds shows where the constant-q assumption breaks.")
else:
    print("VERDICT: residuals look close to white -> the assumed sigma was too small;")
    print("recalibrate the RMSE gate to the measured noise and re-run.")

probe cycle idx 1  n=197  duration 3690 s
  fitted R_int 88.22 mOhm   h 12.48 W/m2K
  residual RMSE      0.7732 K
  lag-1 autocorr     +0.9947   (~0 = white noise, ~1 = structured)
  mean residual by third:  +0.743  -0.397  +0.358  K

VERDICT: residuals are strongly autocorrelated -> STRUCTURAL MISFIT.
The sensor noise is not the problem; the heat-source model is.
Sign pattern across thirds shows where the constant-q assumption breaks.


In [10]:
# ===== CELL 7b : IDENTIFIABILITY OF THE ENRICHED HEAT MODEL =================
# Enrichment: R(x) = R0 (1 + beta * x), x = coulomb-counted depth of discharge
# normalised to [0,1] over the cycle. beta is an EFFECTIVE parameter (b/c): it
# absorbs both the low-SOC resistance rise and the SOC drift of the entropic
# term, which cannot be separated from a single temperature channel.
# Per Section 2: check conditioning BEFORE fitting, never after.

def theta_exp(t, q, h, theta0, L_, rho_cp):
    """Exact solution for piecewise-constant source: dtheta/dt = q/rho_cp - theta/tau."""
    tau = rho_cp*L_/(2.0*h)
    th = np.empty_like(t); th[0] = theta0
    dt = np.diff(t); E = np.exp(-dt/tau)
    for i in range(dt.size):
        th[i+1] = th[i]*E[i] + (q[i]/rho_cp)*tau*(1.0 - E[i])
    return th

def model3(t, R0, beta, h, I, x, T_amb, T0, L_, V_, rho_cp):
    q = I**2 * R0*(1.0 + beta*x) / V_
    return T_amb + theta_exp(t, q, h, T0 - T_amb, L_, rho_cp)

def dod(t, I):
    """Normalised cumulative charge, 0 -> 1 across the cycle."""
    c = np.concatenate([[0.0], np.cumsum(np.abs(I[:-1])*np.diff(t))])
    return c/c[-1] if c[-1] > 0 else np.zeros_like(t)

# representative synthetic cycle matching the stratum's shape
tR = np.linspace(0, 3000.0, 600); IR = np.full_like(tR, 2.0); xR = dod(tR, IR)
TH = dict(R0=0.080, beta=1.5, h=11.0)
T_ambR, T0R = 297.15, 297.15
SIG_R = 0.10          # K, per-sample noise assumed for the bound

def crlb3(names, sigma=SIG_R):
    base = dict(TH)
    cols = []
    for nm in names:
        d = abs(base[nm])*1e-3
        up, dn = dict(base), dict(base)
        up[nm] += d; dn[nm] -= d
        yp = model3(tR, up['R0'], up['beta'], up['h'], IR, xR, T_ambR, T0R, L_NASA, V_NASA, RHOCP_NASA)
        ym = model3(tR, dn['R0'], dn['beta'], dn['h'], IR, xR, T_ambR, T0R, L_NASA, V_NASA, RHOCP_NASA)
        cols.append((yp - ym)/(2*d))
    S = np.vstack(cols)
    F = (S @ S.T)/sigma**2
    sc = np.array([base[n] for n in names])
    condn = np.linalg.cond(F*np.outer(sc, sc))
    try:
        rel = np.sqrt(np.diag(np.linalg.inv(F)))/np.abs(sc)*100
    except np.linalg.LinAlgError:
        rel = np.full(len(names), np.inf)
    return condn, rel

print(f"{'parameter set':28s}{'cond(F) scaled':>16s}   relative CRLB sd, %")
for nms in (["R0", "h"], ["R0", "beta", "h"], ["R0", "beta"]):
    cN, rel = crlb3(nms)
    print(f"{str(nms):28s}{cN:16.3e}   " +
          "  ".join(f"{n}={v:.3f}" for n, v in zip(nms, rel)))
print()
print("Read this before fitting: if {R0, beta, h} is well conditioned the three")
print("are separable from one temperature channel; if not, h must be fixed from")
print("an independent source and only {R0, beta} reported.")

parameter set                 cond(F) scaled   relative CRLB sd, %
['R0', 'h']                        2.433e+02   R0=0.162  h=0.442
['R0', 'beta', 'h']                3.584e+06   R0=0.387  beta=36.793  h=38.916
['R0', 'beta']                     2.263e+02   R0=0.166  beta=0.418

Read this before fitting: if {R0, beta, h} is well conditioned the three
are separable from one temperature channel; if not, h must be fixed from
an independent source and only {R0, beta} reported.


In [11]:
# ===== CELL 7c : ENRICHED FIT ACROSS THE STRATUM, SAME GATES ================
BND3_LO = np.array([1e-4, -0.9, 0.1])
BND3_HI = np.array([1.0, 20.0, 500.0])
MAXPTS  = 600          # subsample long cycles; the model is smooth in time

def fit3(t, y, I, x, T_amb, L_, V_, rho_cp, p0=(0.08, 1.0, 12.0)):
    def m(tt, R0, beta, h):
        return model3(tt, R0, beta, h, I, x, T_amb, y[0], L_, V_, rho_cp)
    popt, pcov = curve_fit(m, t, y, p0=p0, bounds=(BND3_LO, BND3_HI),
                           method='trf', loss='soft_l1', f_scale=0.5)
    return popt, np.sqrt(np.diag(pcov)), m(t, *popt)

rows3 = []
for c in sel:
    t0 = c['t'] - c['t'][0]; y0 = c['T']; I0 = c['I']
    if t0.size > MAXPTS:
        k = np.linspace(0, t0.size-1, MAXPTS).astype(int)
        t0, y0, I0 = t0[k], y0[k], I0[k]
    x0 = dod(t0, I0)
    rise = float(y0.max() - y0[0])
    try:
        (R0, bta, h3), sd3, yf3 = fit3(t0, y0, I0, x0, c['T_amb'], L_NASA, V_NASA, RHOCP_NASA)
        rmse = float(np.sqrt(np.mean((yf3 - y0)**2)))
        p = np.array([R0, bta, h3])
        railed = bool(np.any(np.abs(p - BND3_LO)/np.maximum(np.abs(BND3_LO), 1e-9) < 0.01) or
                      np.any(np.abs(p - BND3_HI)/np.abs(BND3_HI) < 0.01))
        taufrac = float(t0[-1]/(RHOCP_NASA*L_NASA/(2*h3)))
        rows3.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=R0*1e3, beta=bta, h=h3,
                          R_end_mOhm=R0*(1+bta)*1e3, rmse=rmse, rise=rise,
                          tau_frac=taufrac, railed=railed,
                          pass_=(not railed) and rmse <= G_RMSE and rise >= G_RISE
                                and taufrac >= G_TAUFRAC))
    except Exception:
        rows3.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=np.nan, beta=np.nan,
                          h=np.nan, R_end_mOhm=np.nan, rmse=np.nan, rise=rise,
                          tau_frac=np.nan, railed=False, pass_=False))

d3 = pd.DataFrame(rows3); n3 = len(d3)
print(f"enriched model, gate accounting on {n3} cycles")
print(f"  solver raised      : {int(d3.R0_mOhm.isna().sum()):4d}")
print(f"  railed             : {int(d3.railed.sum()):4d}")
print(f"  RMSE > {G_RMSE} K      : {int((d3.rmse > G_RMSE).sum()):4d}")
print(f"  rise  < {G_RISE} K      : {int((d3.rise < G_RISE).sum()):4d}")
print(f"  t/tau < {G_TAUFRAC}        : {int((d3.tau_frac < G_TAUFRAC).sum()):4d}")
g3 = d3[d3.pass_]
print(f"  --> usable          : {len(g3):4d}  ({100*len(g3)/max(n3,1):.1f} %)")
sp = lambda s: (s.quantile(.75)-s.quantile(.25))/s.median() if len(s) else np.nan
if len(g3) >= 3:
    print(f"\nR0     median {g3.R0_mOhm.median():8.2f} mOhm  IQR/median {sp(g3.R0_mOhm):.3f}")
    print(f"beta   median {g3.beta.median():8.3f}        IQR/median {sp(g3.beta):.3f}")
    print(f"R_end  median {g3.R_end_mOhm.median():8.2f} mOhm   (= R0*(1+beta))")
    print(f"h      median {g3.h.median():8.2f} W/m2K  IQR/median {sp(g3.h):.3f}")
    print(f"RMSE   median {g3.rmse.median():8.3f} K")
    cc = g3.dropna(subset=['cap', 'R0_mOhm'])
    if len(cc) > 4:
        print(f"\ncorr(capacity, R0)    = {np.corrcoef(cc.cap, cc.R0_mOhm)[0,1]:+.3f}  (expect NEGATIVE)")
        print(f"corr(capacity, R_end) = {np.corrcoef(cc.cap, cc.R_end_mOhm)[0,1]:+.3f}")
d3.to_csv("nasa_enriched_fits.csv", index=False)
print("\nsaved nasa_enriched_fits.csv")

enriched model, gate accounting on 168 cycles
  solver raised      :    0
  railed             :    0
  RMSE > 0.3 K      :  168
  rise  < 2.0 K      :    0
  t/tau < 0.5        :    0
  --> usable          :    0  (0.0 %)

saved nasa_enriched_fits.csv


### 4.4 The identifiability result, and the two-stage estimator

Cell 7b was run before any three-parameter fitting, and it returned the decisive number:

| Parameter set | scaled cond(F) | relative CRLB sd |
|---|---|---|
| {R₀, h} | 2.43×10² | R₀ 0.16 %, h 0.44 % |
| **{R₀, β, h}** | **3.58×10⁶** | **β 36.8 %, h 38.9 %** |
| {R₀, β}, h known | 2.26×10² | R₀ 0.17 %, β 0.42 % |

**A ramping source and slow cooling bend the temperature curve the same way,** so β and h absorb each other. Fitting all three from one channel on a discharge record is not an estimation problem that a better optimiser can rescue.

The mock confirmed the bound rather than arguing with it: on data generated with β = 1.500 and h = 11.0, the unconstrained three-parameter fit returned **β = 2.305 (+54 %) and h = 17.15 (+56 %)** — both biased in the same direction, along the confounded axis — while achieving an RMSE of **0.051 K**. An excellent trajectory fit with both parameters badly wrong. This is the third independent reproduction of the campaign's central result: **trajectory quality does not certify parameter quality.**

**The two-stage estimator.** h is a property of the fixture — chamber airflow, mounting, surface finish — not of cell state. It should be constant across cycles while R₀ and β age. So:

- **Stage 1 (Cell 7d):** find segments where current is off and the curve is pure exponential decay; fit τ, hence h = ρc_p L / 2τ. No heat-source model is involved, so no confounding is possible.
- **Stage 2 (Cell 7e):** fix h at the stage-1 median and fit {R₀, β} per cycle — the configuration the bound says carries 0.17 % and 0.42 % relative uncertainty.

If no cooling branch exists in a stratum, Cell 7d says so and stops rather than producing three numbers that look like estimates. The fallback order is explicit: pick a stratum that includes the rest period; or concatenate the discharge with the following record's leading rest; or report only the two-parameter fit and state that β is unidentified.

**Mock validation of the full two-stage chain (build container).** Twelve cycles with a 3000 s rest tail, generated at β = 1.500, h = 11.0, R₀ ageing 75.0 → 91.5 mΩ. Recovered: **h = 11.000**, **β = 1.497**, R₀ median 83.15 mΩ, 12/12 usable, RMSE median 0.049 K. Compare against the unconstrained three-parameter result on the same physics: β off by 54 %, h off by 56 %.

**One nuance, recorded rather than smoothed over.** On the same mock *with* the rest tail, the unconstrained three-parameter fit (Cell 7c) also recovered h = 11.00 and β correctly. The confounding is therefore a property of **discharge-only records**, not of the three-parameter model as such: the cooling branch is what supplies the independent information about τ. The two-stage estimator remains preferable because it degrades gracefully — it reports honestly when no cooling branch exists, and it keeps h from drifting cycle-to-cycle on a quantity that physically cannot age — but the underlying requirement is simply that the record must contain a segment where the source is off.

**Harness defect, logged (26 Jul 2026).** Cells 7a–7e initially failed with `NameError` on first real execution, because the mock test harness pre-seeded `L_NASA`, `V_NASA`, `RHOCP_NASA`, `L`, `V_CELL` and `RHO_CP` into its namespace while the notebook defined only the `NASA` dict. The harness was validating physics against a namespace the notebook never actually has. Two defects were masked this way. Fix: Cell 7 now exports the bare names alongside the dict, Cell 7b uses the canonical names, and the regression harness seeds **only** what Cells 2–5 genuinely define — `np`, `curve_fit`, `lumped_T`, `fit_lumped` — so any other missing name surfaces as a real failure. Under that honest namespace the full chain 6 → 6b → 6c → 7 → 7a → 7e → 8 runs clean. Filed under the same category as the earlier "converged 2794/2794" miss: a harness that cannot fail is not a test.

In [12]:
# ===== CELL 7d : STAGE 1 — h FROM THE COOLING BRANCH ========================
# Cell 7b showed {R0, beta, h} is not identifiable from a discharge-only record
# (cond ~ 3.6e6; ~37 % bounds on both beta and h). A ramping source and slow
# cooling bend the curve the same way, so the two absorb each other. The fix is
# decoupling, not a better optimiser.
#
# Physical argument: h is a property of the FIXTURE — chamber airflow, mounting,
# surface finish — not of cell state. It should not change from cycle to cycle,
# while R0 and beta age. So estimate h once, on segments where the current is
# off and the curve is pure exponential decay.

def cooling_segment(c, i_thresh=0.05, min_pts=25, min_drop=0.5):
    """Trailing near-zero-current run, if long enough and actually cooling."""
    off = c['I'] < i_thresh
    if not off.any():
        return None
    idx = np.where(off)[0]
    runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    seg = max(runs, key=len)
    if seg.size < min_pts:
        return None
    t, T = c['t'][seg] - c['t'][seg[0]], c['T'][seg]
    if (T[0] - T[-1]) < min_drop:
        return None
    return t, T

def fit_h_cooling(t, T, T_amb, L_, rho_cp):
    """theta = theta0 exp(-t/tau)  ->  h = rho_cp L /(2 tau)."""
    def m(tt, th0, tau):
        return T_amb + th0*np.exp(-tt/tau)
    popt, _ = curve_fit(m, t, T, p0=[max(T[0]-T_amb, 0.5), 2000.0],
                        bounds=([0.0, 50.0], [200.0, 1e6]),
                        loss='soft_l1', f_scale=0.5)
    th0, tau = popt
    return rho_cp*L_/(2*tau), tau, float(np.sqrt(np.mean((m(t, *popt)-T)**2)))

SPAN_MIN = 0.5      # minimum observed fraction of a time constant
hs = []
for c in sel:
    seg = cooling_segment(c)
    if seg is None:
        continue
    try:
        hh, tau, rr = fit_h_cooling(seg[0], seg[1], c['T_amb'], L_NASA, RHOCP_NASA)
        span = float(seg[0][-1]); frac = span/tau if tau > 0 else 0.0
        # GATE: an exponential observed over << 1 tau is near-linear, so theta0
        # and tau trade off and the fitted tau is truncation-biased.
        if 0.5 < hh < 200 and rr < 1.0:
            hs.append(dict(idx=c['idx'], h=hh, tau=tau, rmse=rr, n=seg[0].size,
                           span=span, span_over_tau=frac, ok=frac >= SPAN_MIN))
    except Exception:
        pass

hd = pd.DataFrame(hs)
n_ok = int(hd.ok.sum()) if len(hd) else 0
print(f"cooling branches found: {len(hd)} of {len(sel)} cycles; "
      f"passing span/tau >= {SPAN_MIN}: {n_ok}")
if len(hd):
    print(f"  span/tau  median {hd.span_over_tau.median():.2f}  "
          f"range [{hd.span_over_tau.min():.2f}, {hd.span_over_tau.max():.2f}]")
if n_ok >= 3:
    hd = hd[hd.ok]
    H_FIX = float(hd.h.median())
    print(hd.head(6).to_string(index=False))
    print(f"\nh  median {H_FIX:.3f} W/m2K   IQR "
          f"[{hd.h.quantile(.25):.3f}, {hd.h.quantile(.75):.3f}]")
    print(f"tau median {hd.tau.median():.0f} s   cooling-fit RMSE median {hd.rmse.median():.3f} K")
    print(f"\nSTAGE 1 COMPLETE: h fixed at {H_FIX:.3f} W/m2K for stage 2.")
else:
    H_FIX = float(hd.h.median()) if len(hd) >= 3 else None
    if H_FIX is not None:
        print(f"\nWARNING: every cooling branch is truncated (span/tau < {SPAN_MIN}).")
        print(f"h = {H_FIX:.3f} W/m2K is reported for continuity but is")
        print("TRUNCATION-BIASED and must not be quoted as a measurement.")
    print("\nNo cooling branch long enough for a clean tau in this stratum.")
    print("Consequence: beta and h cannot be separated (Cell 7b). Options, in order")
    print("of preference:")
    print("  1. choose a stratum whose records include the post-discharge rest;")
    print("  2. concatenate each discharge with the following charge record's")
    print("     leading rest, if the dataset preserves it;")
    print("  3. report only the 2-parameter {R_int, h} fit and treat beta as")
    print("     unidentified — do NOT report a 3-parameter fit as if it were.")

cooling branches found: 97 of 168 cycles; passing span/tau >= 0.5: 24
  span/tau  median 0.42  range [0.26, 0.56]
 idx         h        tau     rmse  n    span  span_over_tau   ok
 512 29.045136 852.122034 0.124155 46 436.437       0.512177 True
 520 28.500288 868.412283 0.121234 46 436.937       0.503145 True
 524 29.103409 850.415841 0.123079 47 446.203       0.524688 True
 528 29.639404 835.037029 0.121111 47 446.187       0.534332 True
 532 29.686940 833.699927 0.117571 48 455.328       0.546153 True
 536 28.686124 862.786467 0.120892 48 456.218       0.528773 True

h  median 29.565 W/m2K   IQR [29.092, 29.679]
tau median 837 s   cooling-fit RMSE median 0.121 K

STAGE 1 COMPLETE: h fixed at 29.565 W/m2K for stage 2.


In [13]:
# ===== CELL 7e : STAGE 2 — {R0, beta} WITH h FIXED ==========================
if H_FIX is None:
    print("Stage 2 skipped: no independent h. See Cell 7d for the options.")
else:
    BND2_LO, BND2_HI = np.array([1e-4, -0.9]), np.array([1.0, 20.0])

    def fit2b(t, y, I, x, T_amb, h_fixed, L_, V_, rho_cp, p0=(0.08, 1.0)):
        def m(tt, R0, beta):
            return model3(tt, R0, beta, h_fixed, I, x, T_amb, y[0], L_, V_, rho_cp)
        popt, pcov = curve_fit(m, t, y, p0=p0, bounds=(BND2_LO, BND2_HI),
                               method='trf', loss='soft_l1', f_scale=0.5)
        return popt, np.sqrt(np.diag(pcov)), m(t, *popt)

    rows2 = []
    for c in sel:
        t0, y0, I0 = c['t'] - c['t'][0], c['T'], c['I']
        if t0.size > MAXPTS:
            k = np.linspace(0, t0.size-1, MAXPTS).astype(int)
            t0, y0, I0 = t0[k], y0[k], I0[k]
        x0 = dod(t0, I0)
        rise = float(y0.max() - y0[0])
        try:
            (R0, bta), sd2, yf2 = fit2b(t0, y0, I0, x0, c['T_amb'], H_FIX,
                                        L_NASA, V_NASA, RHOCP_NASA)
            rmse = float(np.sqrt(np.mean((yf2 - y0)**2)))
            p = np.array([R0, bta])
            railed = bool(np.any(np.abs(p-BND2_LO)/np.maximum(np.abs(BND2_LO), 1e-9) < 0.01)
                          or np.any(np.abs(p-BND2_HI)/np.abs(BND2_HI) < 0.01))
            rows2.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=R0*1e3, beta=bta,
                              R_end_mOhm=R0*(1+bta)*1e3, sd_R0=sd2[0]*1e3,
                              sd_beta=sd2[1], rmse=rmse, rise=rise, railed=railed,
                              pass_=(not railed) and rmse <= G_RMSE and rise >= G_RISE))
        except Exception:
            rows2.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=np.nan, beta=np.nan,
                              R_end_mOhm=np.nan, sd_R0=np.nan, sd_beta=np.nan,
                              rmse=np.nan, rise=rise, railed=False, pass_=False))

    d2 = pd.DataFrame(rows2); n2 = len(d2)
    print(f"stage 2 ({BATTERY} @ {AMBIENT} C, h = {H_FIX:.3f} fixed): {n2} cycles")
    print(f"  solver raised : {int(d2.R0_mOhm.isna().sum()):4d}")
    print(f"  railed        : {int(d2.railed.sum()):4d}")
    print(f"  RMSE > {G_RMSE} K : {int((d2.rmse > G_RMSE).sum()):4d}")
    print(f"  rise  < {G_RISE} K : {int((d2.rise < G_RISE).sum()):4d}")
    g2 = d2[d2.pass_]
    print(f"  --> usable     : {len(g2):4d}  ({100*len(g2)/max(n2,1):.1f} %)")
    sp = lambda s: (s.quantile(.75)-s.quantile(.25))/s.median() if len(s) else np.nan
    if len(g2) >= 3:
        print(f"\nR0     median {g2.R0_mOhm.median():8.2f} mOhm  IQR/median {sp(g2.R0_mOhm):.3f}"
              f"   median fit sd {g2.sd_R0.median():.3f} mOhm")
        print(f"beta   median {g2.beta.median():8.3f}        IQR/median {sp(g2.beta):.3f}"
              f"   median fit sd {g2.sd_beta.median():.3f}")
        print(f"R_end  median {g2.R_end_mOhm.median():8.2f} mOhm")
        print(f"RMSE   median {g2.rmse.median():8.3f} K")
        cc = g2.dropna(subset=['cap', 'R0_mOhm'])
        if len(cc) > 4:
            print(f"\ncorr(capacity, R0)    = {np.corrcoef(cc.cap, cc.R0_mOhm)[0,1]:+.3f}"
                  f"   (expect NEGATIVE)")
            print(f"corr(capacity, R_end) = {np.corrcoef(cc.cap, cc.R_end_mOhm)[0,1]:+.3f}")
        d2.to_csv("nasa_stage2_fits.csv", index=False)
        print("\nsaved nasa_stage2_fits.csv")

stage 2 (B0005 @ 24.0 C, h = 29.565 fixed): 168 cycles
  solver raised :    0
  railed        :    0
  RMSE > 0.3 K :  168
  rise  < 2.0 K :    0
  --> usable     :    0  (0.0 %)


In [14]:
# ===== CELL 7f : RESIDUAL ANATOMY — where does the misfit live? =============
# beta did not reduce RMSE (168/168 still fail) and the residual thirds are
# oscillatory (+,-,+), not monotonic. A monotonic source ramp cannot produce
# that. So locate the misfit before adding any further parameter.

RMSE_1NODE = {}
def anatomy(c, R, h, beta=None, label=""):
    t = c['t'] - c['t'][0]; y = c['T']; I = c['I']
    if beta is None:
        I_fun = lambda tt, tt_=t, I_=I: float(np.interp(tt, tt_, I_))
        yf = lumped_T(t, R, h, RHOCP_NASA, I_fun, t[-1], y[0], c['T_amb'], L_NASA, V_NASA)
    else:
        yf = model3(t, R, beta, h, I, dod(t, I), c['T_amb'], y[0],
                    L_NASA, V_NASA, RHOCP_NASA)
    r = y - yf
    on, off = I > 0.05, I <= 0.05
    RMSE_1NODE[label.strip()] = float(np.sqrt(np.mean(r**2)))
    print(f"  {label:28s} RMSE {np.sqrt(np.mean(r**2)):.3f} K", end="")
    if on.any():
        print(f" | load {np.sqrt(np.mean(r[on]**2)):.3f} ({on.sum()} pts)", end="")
    if off.any():
        print(f" | rest {np.sqrt(np.mean(r[off]**2)):.3f} ({off.sum()} pts)", end="")
    print()
    return t, r, on, off

print(f"probe cycle {probe['idx']}: n={probe['t'].size}, "
      f"duration {probe['t'][-1]-probe['t'][0]:.0f} s")
non_zero = probe['I'] > 0.05
print(f"  load samples {int(non_zero.sum())}, rest samples {int((~non_zero).sum())}")
print()
t_a, r_a, on_a, off_a = anatomy(probe, Rp, hp, None, "1-node, constant q")
row = d3[d3.idx == probe['idx']] if 'd3' in dir() else None
if row is not None and len(row) and np.isfinite(row.R0_mOhm.iloc[0]):
    anatomy(probe, row.R0_mOhm.iloc[0]/1e3, row.h.iloc[0], row.beta.iloc[0],
            "1-node, ramped q (beta)")

print("\n  residual by decile of elapsed time (K):")
q = np.linspace(0, len(r_a), 11).astype(int)
print("   ", "  ".join(f"{r_a[q[i]:q[i+1]].mean():+.2f}" for i in range(10)))
print("\n  sign changes in the binned residual:",
      int(np.sum(np.diff(np.sign([r_a[q[i]:q[i+1]].mean() for i in range(10)])) != 0)))
print("  A monotonic source error gives 0-1 sign changes; more than that means")
print("  the DYNAMICS are wrong, not the source amplitude.")

probe cycle 1: n=197, duration 3690 s
  load samples 178, rest samples 19

  1-node, constant q           RMSE 0.773 K | load 0.726 (178 pts) | rest 1.124 (19 pts)
  1-node, ramped q (beta)      RMSE 0.696 K | load 0.569 (178 pts) | rest 1.408 (19 pts)

  residual by decile of elapsed time (K):
    +0.42  +1.17  +0.76  +0.10  -0.29  -0.60  -0.68  -0.28  +0.75  +0.96

  sign changes in the binned residual: 2
  A monotonic source error gives 0-1 sign changes; more than that means
  the DYNAMICS are wrong, not the source amplitude.


In [15]:
# ===== CELL 7g : TWO-NODE STRUCTURAL PROBE ==================================
# Question asked here is deliberately NOT "what are the parameters" but
# "is the single-node STRUCTURE capable of fitting this data at all".
# So the probe is allowed more freedom than an estimator would be; if it still
# cannot reach the gate, the structure is refuted and no reparameterisation of
# a single-node model will help.
#
# Physical motivation: Part 2 assumed k = 2.5 W/(m K) (effective, flagged
# moderate confidence). The RADIAL conductivity of a wound 18650 is ~0.2-0.5
# W/(m K) (b) because the jelly-roll layers act as a thermal barrier. With the
# stage-1 h that puts Bi = h r / k in 0.5-1.3, NOT << 1: the surface sensor
# then reads a lagged, attenuated version of the core.
#
#   C_c dT_c/dt = Q(t) - (T_c - T_s)/R_cs
#   C_s dT_s/dt = (T_c - T_s)/R_cs - (T_s - T_amb)/R_sa      [measured: T_s]

A_CELL = 2*np.pi*0.009*0.065 + 2*np.pi*0.009**2      # m^2, 18650 outer area
C_TOT  = RHOCP_NASA*V_NASA                            # J/K, total heat capacity (b)

def two_node(t, f, R_cs, h_s, R0, beta, I, x, T_amb, T0, dt_max=2.0):
    Cc, Cs = f*C_TOT, (1.0-f)*C_TOT
    n = max(int((t[-1]-t[0])/dt_max), len(t)*4)
    tu = np.linspace(t[0], t[-1], n)
    Iu = np.interp(tu, t, I); xu = np.interp(tu, t, x)
    Q = Iu**2*R0*(1.0 + beta*xu)
    Tc = np.full(n, T0); Ts = np.full(n, T0)
    dt = tu[1]-tu[0]
    for i in range(n-1):
        q_cs = (Tc[i]-Ts[i])/R_cs
        Tc[i+1] = Tc[i] + dt*(Q[i] - q_cs)/Cc
        Ts[i+1] = Ts[i] + dt*(q_cs - h_s*A_CELL*(Ts[i]-T_amb))/Cs
    return np.interp(t, tu, Ts)

def fit_two_node(c):
    t = c['t'] - c['t'][0]; y = c['T']; I = c['I']; x = dod(t, I)
    def m(tt, f, R_cs, h_s, R0, beta):
        return two_node(tt, f, R_cs, h_s, R0, beta, I, x, c['T_amb'], y[0])
    # PHYSICAL bounds. Wide bounds let a 5-parameter probe escape to nonsense
    # (R0 -> 1 ohm, h -> 300 W/m2K) and mimic a source error, which destroys the
    # probe's ability to testify about structure.
    #   R0   20-300 mOhm  : 18650 DCIR range (b)
    #   h    2-100 W/m2K  : natural to mild forced convection (b)
    #   R_cs 0.5-20 K/W   : 1/(4 pi k H) for k = 0.06-2.5 W/(m K), H = 65 mm (b)
    p0 = (0.7, 3.0, 20.0, 0.08, 1.0)
    lo = (0.10, 0.5, 2.0, 0.020, -0.9)
    hi = (0.95, 20.0, 100.0, 0.300, 10.0)
    popt, _ = curve_fit(m, t, y, p0=p0, bounds=(lo, hi), method='trf',
                        loss='soft_l1', f_scale=0.5, max_nfev=4000)
    yf = m(t, *popt)
    return popt, float(np.sqrt(np.mean((yf-y)**2)))

try:
    LO = np.array([0.10, 0.5, 2.0, 0.020, -0.9])
    HI = np.array([0.95, 20.0, 100.0, 0.300, 10.0])
    (f_, Rcs_, hs_, R0_, b_), rmse2 = fit_two_node(probe)
    p = np.array([f_, Rcs_, hs_, R0_, b_])
    railed2 = bool(np.any(np.abs(p-LO)/np.maximum(np.abs(LO), 1e-9) < 0.01) or
                   np.any(np.abs(p-HI)/np.abs(HI) < 0.01))
    tau_int = f_*C_TOT*Rcs_
    k_eff = 0.009/(Rcs_*A_CELL) if Rcs_ > 0 else np.nan
    best1 = min(RMSE_1NODE.values())
    best1_name = min(RMSE_1NODE, key=RMSE_1NODE.get)
    print(f"two-node probe on cycle {probe['idx']}:")
    print(f"  core fraction f   {f_:.3f}   (C_core = {f_*C_TOT:.1f} J/K of {C_TOT:.1f})")
    print(f"  R_cs              {Rcs_:.3f} K/W   -> internal tau {tau_int:.0f} s")
    print(f"  h_surface         {hs_:.2f} W/m2K")
    print(f"  R0, beta          {R0_*1e3:.2f} mOhm, {b_:.3f}")
    print(f"  implied radial k  {k_eff:.3f} W/(m K)   (b; Part-2 assumed 2.5)")
    print(f"  railed            {railed2}")
    print()
    print(f"  best single-node  {best1:.3f} K  ({best1_name})")
    print(f"  two-node          {rmse2:.3f} K")
    print()
    if railed2:
        print("INCONCLUSIVE: the two-node probe railed on a bound. A railed 5-parameter")
        print("fit can mimic a source error, so it cannot testify about structure.")
        print("Widen the bound that railed and re-run, or reduce the probe's freedom.")
    elif best1 <= G_RMSE:
        print("NO STRUCTURAL PROBLEM: a single-node model already clears the gate on this")
        print("cycle. The two-node probe is unnecessary here.")
    elif rmse2 <= G_RMSE and rmse2 < 0.5*best1:
        print("STRUCTURE REFUTED for the single-node model: two-node clears the gate on")
        print("the same data while the best single-node model does not, and the")
        print("improvement is more than 2x. The missing physics is the internal")
        print("gradient, not the heat source. Next: identifiability of the two-node")
        print("set, then the radial 1-D inverse - the thesis direction anyway.")
    elif rmse2 < best1:
        print(f"PARTIAL: two-node improves ({100*(1-rmse2/best1):.0f} %) but does not clear")
        print("the gate. Something beyond the internal gradient is also missing -")
        print("check ambient drift, sensor placement and record segmentation.")
    else:
        print("NO IMPROVEMENT: the internal gradient is not the missing physics.")
        print("Do not add nodes. Investigate ambient drift and record segmentation.")
except Exception as e:
    print("two-node probe failed:", type(e).__name__, e)


two-node probe on cycle 1:
  core fraction f   0.168   (C_core = 7.7 J/K of 45.5)
  R_cs              19.995 K/W   -> internal tau 153 s
  h_surface         35.12 W/m2K
  R0, beta          205.01 mOhm, 1.605
  implied radial k  0.108 W/(m K)   (b; Part-2 assumed 2.5)
  railed            True

  best single-node  0.696 K  (1-node, ramped q (beta))
  two-node          0.725 K

INCONCLUSIVE: the two-node probe railed on a bound. A railed 5-parameter
fit can mimic a source error, so it cannot testify about structure.
Widen the bound that railed and re-run, or reduce the probe's freedom.


In [16]:
# ===== CELL 7h : HOW MUCH SOURCE SHAPE DOES THE DATA SUPPORT? ==============
# Cell 7f: 2 sign changes in the binned residual, data hotter EARLY and LATE,
# cooler MID. A monotonic ramp cannot bend twice. Physically expected for LCO:
# DCIR is U-shaped against SOC (high at full charge, minimum mid, climbing
# steeply toward cutoff) and the entropic term changes sign with SOC. Neither
# is assumed numerically here -- the shape is INFERRED, which is the inverse
# problem this project exists to solve.
#
#   R(x) = R0 * (1 + c1 x + c2 x^2 + ... + cn x^n),   x = depth of discharge
#
# Identifiability first, as in Section 2: adding shape freedom always improves
# the fit, so the question is how many coefficients the data can actually
# support before the CRLB blows up.

def model_poly(t, R0, coefs, h, I, x, T_amb, T0, L_, V_, rho_cp):
    mult = np.ones_like(x)
    for k, ck in enumerate(coefs, start=1):
        mult = mult + ck*x**k
    q = I**2*R0*mult/V_
    return T_amb + theta_exp(t, q, h, T0 - T_amb, L_, rho_cp)

# representative cycle geometry taken from the probe so the bound is realistic
_t = probe['t'] - probe['t'][0]
_I = probe['I']; _x = dod(_t, _I)
_Tamb, _T0 = probe['T_amb'], probe['T'][0]
SIG_H = 0.10        # K per sample; the bound scales as 1/sigma

def crlb_poly(order, h_val, fit_h=False, R0=0.09, sigma=SIG_H):
    base = [R0] + [0.5]*order + ([h_val] if fit_h else [])
    names = ["R0"] + [f"c{k}" for k in range(1, order+1)] + (["h"] if fit_h else [])
    def ev(p):
        R0_, cs = p[0], p[1:1+order]
        hh = p[1+order] if fit_h else h_val
        return model_poly(_t, R0_, cs, hh, _I, _x, _Tamb, _T0, L_NASA, V_NASA, RHOCP_NASA)
    cols = []
    for j in range(len(base)):
        d = abs(base[j])*1e-3 or 1e-6
        up, dn = list(base), list(base)
        up[j] += d; dn[j] -= d
        cols.append((ev(up) - ev(dn))/(2*d))
    S = np.vstack(cols); F = (S @ S.T)/sigma**2
    sc = np.array(base)
    condn = np.linalg.cond(F*np.outer(sc, sc))
    try:
        rel = np.sqrt(np.diag(np.linalg.inv(F)))/np.abs(sc)*100
    except np.linalg.LinAlgError:
        rel = np.full(len(base), np.inf)
    return names, condn, rel

H_REF = H_FIX if ('H_FIX' in dir() and H_FIX) else hp
print(f"source-shape identifiability at h = {H_REF:.2f} W/m2K, sigma = {SIG_H} K")
print(f"{'model':34s}{'cond(F)':>12s}   worst relative CRLB sd, %")
SUPPORTED = 1
for order in (1, 2, 3, 4):
    for fit_h in (False, True):
        nm, cN, rel = crlb_poly(order, H_REF, fit_h)
        tag = f"order {order}" + (" + h free" if fit_h else " , h fixed")
        print(f"{tag:34s}{cN:12.3e}   {np.max(rel):8.2f}   ({nm[int(np.argmax(rel))]})")
        if (not fit_h) and cN < 1e5 and np.max(rel) < 25:
            SUPPORTED = max(SUPPORTED, order)
print(f"\nHighest order supported with h fixed (cond < 1e5 and all sd < 25 %): {SUPPORTED}")
print("Above that the coefficients trade off and the fit will look better while")
print("meaning less -- the same trap Cell 7b caught for beta and h.")

source-shape identifiability at h = 29.57 W/m2K, sigma = 0.1 K
model                                  cond(F)   worst relative CRLB sd, %
order 1 , h fixed                    2.412e+02       1.63   (c1)
order 1 + h free                     3.049e+03       4.38   (c1)
order 2 , h fixed                    1.070e+04       7.48   (c1)
order 2 + h free                     1.805e+04       8.43   (c1)
order 3 , h fixed                    6.136e+05      52.69   (c2)
order 3 + h free                     1.211e+06      59.35   (c2)
order 4 , h fixed                    3.269e+07     362.97   (c3)
order 4 + h free                     5.285e+07     377.01   (c3)

Highest order supported with h fixed (cond < 1e5 and all sd < 25 %): 2
Above that the coefficients trade off and the fit will look better while
meaning less -- the same trap Cell 7b caught for beta and h.


In [17]:
# ===== CELL 7i : POLYNOMIAL-SOURCE FIT ACROSS THE STRATUM ==================
ORDER = SUPPORTED            # set manually to override
print(f"fitting source polynomial of order {ORDER} with h "
      + (f"fixed at {H_FIX:.3f}" if ('H_FIX' in dir() and H_FIX) else "free"))

def fit_poly(t, y, I, x, T_amb, h_fixed, order, L_, V_, rho_cp):
    lo = [0.020] + [-20.0]*order
    hi = [0.300] + [ 20.0]*order
    p0 = [0.09] + [0.5]*order
    if h_fixed is None:
        lo, hi, p0 = lo+[2.0], hi+[100.0], p0+[15.0]
    def m(tt, *p):
        R0_, cs = p[0], list(p[1:1+order])
        hh = p[1+order] if h_fixed is None else h_fixed
        return model_poly(tt, R0_, cs, hh, I, x, T_amb, y[0], L_, V_, rho_cp)
    popt, pcov = curve_fit(m, t, y, p0=p0, bounds=(lo, hi), method='trf',
                           loss='soft_l1', f_scale=0.5, max_nfev=6000)
    railed = bool(np.any(np.abs(popt-np.array(lo))/np.maximum(np.abs(lo), 1e-9) < 0.01)
                  or np.any(np.abs(popt-np.array(hi))/np.abs(hi) < 0.01))
    return popt, np.sqrt(np.diag(pcov)), m(t, *popt), railed

H_USE = H_FIX if ('H_FIX' in dir() and H_FIX) else None
rows = []
for c in sel:
    t0, y0, I0 = c['t'] - c['t'][0], c['T'], c['I']
    if t0.size > MAXPTS:
        k = np.linspace(0, t0.size-1, MAXPTS).astype(int)
        t0, y0, I0 = t0[k], y0[k], I0[k]
    x0 = dod(t0, I0); rise = float(y0.max()-y0[0])
    try:
        popt, sd, yf, railed = fit_poly(t0, y0, I0, x0, c['T_amb'], H_USE, ORDER,
                                        L_NASA, V_NASA, RHOCP_NASA)
        rmse = float(np.sqrt(np.mean((yf-y0)**2)))
        rec = dict(idx=c['idx'], cap=c['cap'], R0_mOhm=popt[0]*1e3, rmse=rmse,
                   rise=rise, railed=railed,
                   pass_=(not railed) and rmse <= G_RMSE and rise >= G_RISE)
        for k_ in range(ORDER):
            rec[f"c{k_+1}"] = popt[1+k_]
        # heat multiplier at the two ends of the discharge
        mult_end = 1.0 + sum(popt[1+k_] for k_ in range(ORDER))
        rec["R_end_mOhm"] = popt[0]*mult_end*1e3
        rows.append(rec)
    except Exception:
        rows.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=np.nan, rmse=np.nan,
                         rise=rise, railed=False, pass_=False))

dp = pd.DataFrame(rows); n = len(dp)
print(f"\ngate accounting on {n} cycles")
print(f"  solver raised : {int(dp.R0_mOhm.isna().sum()):4d}")
print(f"  railed        : {int(dp.railed.sum()):4d}")
print(f"  RMSE > {G_RMSE} K : {int((dp.rmse > G_RMSE).sum()):4d}")
gp = dp[dp.pass_]
print(f"  --> usable     : {len(gp):4d}  ({100*len(gp)/max(n,1):.1f} %)")
sp = lambda s: (s.quantile(.75)-s.quantile(.25))/s.median() if len(s) else np.nan
if len(gp) >= 3:
    print(f"\nR0     median {gp.R0_mOhm.median():8.2f} mOhm  IQR/median {sp(gp.R0_mOhm):.3f}")
    for k_ in range(ORDER):
        col = f"c{k_+1}"
        print(f"{col:6s} median {gp[col].median():8.3f}")
    print(f"R_end  median {gp.R_end_mOhm.median():8.2f} mOhm")
    print(f"RMSE   median {gp.rmse.median():8.3f} K   (single-node ramp was ~0.70 K)")
    cc = gp.dropna(subset=['cap', 'R0_mOhm'])
    if len(cc) > 4:
        print(f"\ncorr(capacity, R0)    = {np.corrcoef(cc.cap, cc.R0_mOhm)[0,1]:+.3f}  (expect NEGATIVE)")
        print(f"corr(capacity, R_end) = {np.corrcoef(cc.cap, cc.R_end_mOhm)[0,1]:+.3f}")
    dp.to_csv("nasa_poly_fits.csv", index=False)
    print("\nsaved nasa_poly_fits.csv")
else:
    print("\nStill failing. Next candidate is ambient/fixture drift: T_amb is a")
    print("per-cycle scalar, not a logged trace. Add T_amb as a free parameter and")
    print("re-check identifiability BEFORE trusting any resulting number.")

fitting source polynomial of order 2 with h fixed at 29.565



gate accounting on 168 cycles
  solver raised :    0
  railed        :    0
  RMSE > 0.3 K :    9
  --> usable     :  159  (94.6 %)

R0     median   157.48 mOhm  IQR/median 0.161
c1     median   -1.255
c2     median    2.276
R_end  median   306.78 mOhm
RMSE   median    0.260 K   (single-node ramp was ~0.70 K)

corr(capacity, R0)    = -0.906  (expect NEGATIVE)
corr(capacity, R_end) = -0.986

saved nasa_poly_fits.csv


In [18]:
# ===== CELL 7j : DID THE OSCILLATION ACTUALLY GO AWAY? ======================
# Extra freedom always lowers RMSE. The test that the RIGHT freedom was added is
# that the STRUCTURE of the residual disappears: the 2 sign changes seen in
# Cell 7f should collapse to 0-1, and the load/rest imbalance should shrink.

def decile_profile(r, nb_=10):
    q = np.linspace(0, len(r), nb_+1).astype(int)
    return np.array([r[q[i]:q[i+1]].mean() for i in range(nb_)])

t0, y0, I0 = probe['t']-probe['t'][0], probe['T'], probe['I']
if t0.size > MAXPTS:
    k = np.linspace(0, t0.size-1, MAXPTS).astype(int); t0, y0, I0 = t0[k], y0[k], I0[k]
x0 = dod(t0, I0)
popt_p, sd_p, yf_p, railed_p = fit_poly(t0, y0, I0, x0, probe['T_amb'],
                                        H_USE, ORDER, L_NASA, V_NASA, RHOCP_NASA)
r_p = y0 - yf_p
on0 = I0 > 0.05; off0 = ~on0
d_new = decile_profile(r_p)
sc_new = int(np.sum(np.diff(np.sign(d_new)) != 0))
ac_new = float(np.sum(r_p[:-1]*r_p[1:])/np.sum(r_p**2))

print(f"probe cycle {probe['idx']}, order-{ORDER} source, h "
      + (f"= {H_USE:.3f} fixed" if H_USE else "free"))
print(f"  RMSE        {np.sqrt(np.mean(r_p**2)):.3f} K", end="")
if on0.any():  print(f" | load {np.sqrt(np.mean(r_p[on0]**2)):.3f}", end="")
if off0.any(): print(f" | rest {np.sqrt(np.mean(r_p[off0]**2)):.3f}", end="")
print()
print(f"  lag-1 autocorr {ac_new:+.4f}   (was +0.9947 with constant q)")
print("  residual by decile (K):")
print("   ", "  ".join(f"{v:+.2f}" for v in d_new))
struct = float(np.std(d_new)/np.sqrt(np.mean(r_p**2)))
print(f"  sign changes  {sc_new}   structure ratio std(deciles)/RMSE = {struct:.2f}")
print()
print("  NOTE ON METRICS: sign-change COUNT alone is misleading - white noise")
print("  crosses zero often while a structured error crosses rarely but with large")
print("  excursions. The discriminators are the lag-1 autocorrelation and the")
print("  structure ratio. Reference values: constant-q on real data gave")
print("  autocorr +0.99 and ratio 0.83; a correctly specified model on mock data")
print("  gave autocorr +0.05 and ratio 0.37.")
print()
if ac_new < 0.5 and struct < 0.5:
    print("STRUCTURE CLEARED: the residual is close to unstructured. The non-monotonic")
    print("source was the right diagnosis - the mechanism was explained, not absorbed.")
elif ac_new < 0.85:
    print("PARTIALLY CLEARED: structure much reduced but not gone. Do NOT raise the")
    print("polynomial order (Cell 7h says 3 is unsupported). Next candidate is")
    print("ambient/fixture drift - add T_amb as a free parameter, re-check the CRLB")
    print("first, and only then trust any number that comes out.")
else:
    print("STRUCTURE PERSISTS at close to its original strength: the extra freedom was")
    print("absorbed rather than a mechanism explained. Re-open the diagnosis.")

probe cycle 1, order-2 source, h = 29.565 fixed
  RMSE        0.145 K | load 0.131 | rest 0.238
  lag-1 autocorr +0.9858   (was +0.9947 with constant q)
  residual by decile (K):
    -0.09  +0.07  -0.06  -0.11  +0.09  +0.14  +0.03  -0.11  -0.21  +0.23
  sign changes  5   structure ratio std(deciles)/RMSE = 0.89

  NOTE ON METRICS: sign-change COUNT alone is misleading - white noise
  crosses zero often while a structured error crosses rarely but with large
  excursions. The discriminators are the lag-1 autocorrelation and the
  structure ratio. Reference values: constant-q on real data gave
  autocorr +0.99 and ratio 0.83; a correctly specified model on mock data
  gave autocorr +0.05 and ratio 0.37.

STRUCTURE PERSISTS at close to its original strength: the extra freedom was
absorbed rather than a mechanism explained. Re-open the diagnosis.


In [19]:
# ===== CELL 7k : MODEL ADEQUACY AGAINST THE MEASURED NOISE FLOOR ============
# Cell 7j judged "structure cleared" using an autocorrelation threshold
# calibrated on a mock with i.i.d. Gaussian noise. That reference is
# unreachable on real data: autocorrelation is SCALE-FREE, and real thermocouple
# noise is correlated (junction thermal mass, ADC filtering, quantisation), so a
# densely sampled residual always looks autocorrelated no matter how small it is.
#
# The scale-aware test: estimate the sensor noise floor from the data itself and
# ask whether the residual has been driven down to it.
#
# Second-difference estimator: for a smooth signal plus white noise of sd sigma,
#   d2[i] = y[i+2] - 2 y[i+1] + y[i]   has variance 6 sigma^2.
# The signal's own curvature inflates it, so this is an UPPER bound on sigma --
# which makes the adequacy ratio below conservative in the safe direction.

def noise_floor(y):
    d2 = y[2:] - 2*y[1:-1] + y[:-2]
    return float(np.std(d2)/np.sqrt(6.0))

sig_hat = noise_floor(probe['T'])
rmse_now = float(np.sqrt(np.mean(r_p**2)))
ratio = rmse_now/sig_hat if sig_hat > 0 else np.inf
print(f"probe cycle {probe['idx']}")
print(f"  measured noise floor (2nd-difference, upper bound)  {sig_hat:.4f} K")
print(f"  residual RMSE, order-{ORDER} source                  {rmse_now:.4f} K")
print(f"  adequacy ratio RMSE / sigma                        {ratio:.2f}")
print()
# same for the constant-q baseline, for contrast
if 'r_a' in dir():
    print(f"  for contrast, constant-q residual RMSE was        "
          f"{np.sqrt(np.mean(r_a**2)):.4f} K  (ratio {np.sqrt(np.mean(r_a**2))/sig_hat:.2f})")
print()
if ratio <= 1.5:
    print("AT THE NOISE FLOOR: the residual is as small as the instrument allows.")
    print("No further model change is justified by this cycle - any improvement")
    print("would be fitting the sensor, not the physics.")
elif ratio <= 3.0:
    print("CLOSE TO THE NOISE FLOOR: the dominant mechanism has been captured.")
    print("What remains is within a small multiple of the measurement limit, so")
    print("further structure is worth chasing only if it is systematic across")
    print("cycles AND concentrated in one phase - check the load/rest split.")
else:
    print("WELL ABOVE THE NOISE FLOOR: real structure remains. Identify WHICH")
    print("phase carries it before adding parameters.")

# where does what remains live?
on_p = probe['I'][:r_p.size] > 0.05 if probe['I'].size >= r_p.size else None
print()
print("Reminder from Cell 7j on this cycle: load 0.131 K vs rest 0.238 K.")
print("The rest phase is where h acts alone, and the stage-1 h came from cooling")
print("branches spanning only ~0.28 tau. Cell 7l estimates h from all cycles at")
print("once instead, which is the cheapest available fix.")

probe cycle 1
  measured noise floor (2nd-difference, upper bound)  0.0120 K
  residual RMSE, order-2 source                  0.1448 K
  adequacy ratio RMSE / sigma                        12.05

  for contrast, constant-q residual RMSE was        0.7732 K  (ratio 64.37)

WELL ABOVE THE NOISE FLOOR: real structure remains. Identify WHICH
phase carries it before adding parameters.

Reminder from Cell 7j on this cycle: load 0.131 K vs rest 0.238 K.
The rest phase is where h acts alone, and the stage-1 h came from cooling
branches spanning only ~0.28 tau. Cell 7l estimates h from all cycles at
once instead, which is the cheapest available fix.


In [20]:
# ===== CELL 7l : PROFILE LIKELIHOOD FOR A SHARED h =========================
# h is a property of the FIXTURE - chamber airflow, mounting, surface finish -
# so it is the same for every cycle in a stratum, while R0 and the source shape
# age. No single cycle here carries a cooling branch long enough to pin tau
# (span/tau ~ 0.28), but 168 cycles constrain a SHARED h jointly.
#
# Method: for each trial h, refit {R0, c1, ..., cn} on every cycle with that h
# held, and accumulate the total residual. The curve of total RMSE against h is
# a profile likelihood. A clear minimum means h is identifiable; a flat curve
# means it is not, and says so honestly instead of returning a number.

H_GRID = np.array([8, 10, 12, 15, 18, 22, 26, 30, 35, 40, 46, 52, 60], float)
STRIDE = max(1, len(sel)//28)          # subsample cycles for speed
SUB = sel[::STRIDE]
print(f"profiling h over {len(H_GRID)} values using {len(SUB)} of {len(sel)} cycles")

prof = []
for hh in H_GRID:
    sse, npts, nok = 0.0, 0, 0
    for c in SUB:
        t0, y0, I0 = c['t']-c['t'][0], c['T'], c['I']
        if t0.size > MAXPTS:
            k = np.linspace(0, t0.size-1, MAXPTS).astype(int)
            t0, y0, I0 = t0[k], y0[k], I0[k]
        x0 = dod(t0, I0)
        try:
            popt, sd, yf, railed = fit_poly(t0, y0, I0, x0, c['T_amb'], hh, ORDER,
                                            L_NASA, V_NASA, RHOCP_NASA)
            if not railed:
                sse += float(np.sum((yf-y0)**2)); npts += y0.size; nok += 1
        except Exception:
            pass
    prof.append(np.sqrt(sse/npts) if npts else np.nan)
    print(f"  h = {hh:5.1f} W/m2K   total RMSE {prof[-1]:.4f} K   ({nok} cycles)")

prof = np.array(prof)
ok = np.isfinite(prof)
if ok.sum() >= 4:
    i_min = int(np.nanargmin(prof))
    h_best = H_GRID[i_min]
    depth = (np.nanmax(prof)-np.nanmin(prof))/np.nanmin(prof)
    print(f"\n  minimum at h = {h_best:.1f} W/m2K, total RMSE {prof[i_min]:.4f} K")
    print(f"  profile depth (max-min)/min = {depth:.3f}")
    # parabolic refinement and a 1-sigma-style interval from the curvature
    if 0 < i_min < len(H_GRID)-1:
        a, b, c_ = prof[i_min-1], prof[i_min], prof[i_min+1]
        denom = (a - 2*b + c_)
        if denom > 0:
            shift = 0.5*(a - c_)/denom
            h_ref = h_best + shift*(H_GRID[i_min+1]-H_GRID[i_min-1])/2
            print(f"  parabolic refinement -> h = {h_ref:.2f} W/m2K")
    print()
    if depth < 0.05:
        print("PROFILE IS FLAT: h is not identifiable from these discharge records")
        print("even pooled. Report the source shape and treat absolute R0 as")
        print("provisional; to fix h you need a real cooling branch - concatenate")
        print("each discharge with the following record's leading rest.")
    else:
        print("h IS IDENTIFIABLE from the pooled data. Set H_FIX to the value above,")
        print("re-run Cell 7i, and absolute R0 becomes quotable rather than provisional.")
        print(f"  compare: stage-1 cooling-branch estimate was "
              + (f"{H_FIX:.2f}" if ('H_FIX' in dir() and H_FIX) else "unavailable")
              + " W/m2K (truncation-biased).")
else:
    print("\nprofile failed - too few converged fits")

profiling h over 13 values using 28 of 168 cycles


  h =   8.0 W/m2K   total RMSE 0.5298 K   (28 cycles)


  h =  10.0 W/m2K   total RMSE 0.4746 K   (28 cycles)


  h =  12.0 W/m2K   total RMSE 0.4223 K   (28 cycles)


  h =  15.0 W/m2K   total RMSE 0.3508 K   (28 cycles)


...

  h =  35.0 W/m2K   total RMSE 0.3513 K   (28 cycles)


  h =  40.0 W/m2K   total RMSE 0.4624 K   (28 cycles)


  h =  46.0 W/m2K   total RMSE 0.6091 K   (28 cycles)


  h =  52.0 W/m2K   total RMSE 0.7578 K   (28 cycles)


  h =  60.0 W/m2K   total RMSE 0.9489 K   (28 cycles)

  minimum at h = 26.0 W/m2K, total RMSE 0.2296 K
  profile depth (max-min)/min = 3.132
  parabolic refinement -> h = 24.81 W/m2K

h IS IDENTIFIABLE from the pooled data. Set H_FIX to the value above,
re-run Cell 7i, and absolute R0 becomes quotable rather than provisional.
  compare: stage-1 cooling-branch estimate was 29.57 W/m2K (truncation-biased).


In [21]:
# ===== CELL 7m : FINAL FIT AT THE PROFILE-LIKELIHOOD h =====================
# Cell 7l located h from all cycles jointly. Use it: this refit produces the
# numbers that may actually be quoted, because h no longer carries the
# cooling-branch truncation bias.
H_PROFILE = 24.81          # <- set from Cell 7l's parabolic refinement
print(f"refitting {len(sel)} cycles at h = {H_PROFILE} W/m2K (profile likelihood), "
      f"order {ORDER}")
print(f"  previous run used h = "
      + (f"{H_FIX:.3f}" if ('H_FIX' in dir() and H_FIX) else "n/a")
      + " from truncated cooling branches\n")

rows = []
for c in sel:
    t0, y0, I0 = c['t']-c['t'][0], c['T'], c['I']
    if t0.size > MAXPTS:
        k = np.linspace(0, t0.size-1, MAXPTS).astype(int)
        t0, y0, I0 = t0[k], y0[k], I0[k]
    x0 = dod(t0, I0); rise = float(y0.max()-y0[0])
    try:
        popt, sd, yf, railed = fit_poly(t0, y0, I0, x0, c['T_amb'], H_PROFILE,
                                        ORDER, L_NASA, V_NASA, RHOCP_NASA)
        rmse = float(np.sqrt(np.mean((yf-y0)**2)))
        mult_end = 1.0 + sum(popt[1+k_] for k_ in range(ORDER))
        rec = dict(idx=c['idx'], cap=c['cap'], R0_mOhm=popt[0]*1e3,
                   R_end_mOhm=popt[0]*mult_end*1e3, rmse=rmse, rise=rise,
                   sigma=noise_floor(c['T']), railed=railed,
                   pass_=(not railed) and rmse <= G_RMSE and rise >= G_RISE)
        for k_ in range(ORDER):
            rec[f"c{k_+1}"] = popt[1+k_]
        rows.append(rec)
    except Exception:
        rows.append(dict(idx=c['idx'], cap=c['cap'], R0_mOhm=np.nan, rmse=np.nan,
                         rise=rise, railed=False, pass_=False))

dm = pd.DataFrame(rows); gm = dm[dm.pass_]
sp = lambda s: (s.quantile(.75)-s.quantile(.25))/s.median() if len(s) else np.nan
print(f"usable {len(gm)}/{len(dm)} ({100*len(gm)/max(len(dm),1):.1f} %), "
      f"railed {int(dm.railed.sum())}")
if len(gm) >= 3:
    print(f"\nR0     median {gm.R0_mOhm.median():8.2f} mOhm  IQR/median {sp(gm.R0_mOhm):.3f}")
    # Report the SPREAD for the shape coefficients too. Earlier versions gave
    # R0 an IQR and c1/c2 as bare medians -- an asymmetry that made the shape
    # look more certain than it had been shown to be.
    for k_ in range(ORDER):
        col = gm[f'c{k_+1}']
        print(f"c{k_+1}     median {col.median():8.3f}  IQR "
              f"[{col.quantile(.25):.3f}, {col.quantile(.75):.3f}]  "
              f"IQR/|median| {abs(sp(col)):.3f}")
    if ORDER == 2:
        xm = -gm.c1/(2*gm.c2)
        print(f"x_min  median {xm.median():8.3f}  IQR "
              f"[{xm.quantile(.25):.3f}, {xm.quantile(.75):.3f}]"
              f"   (location of the DCIR minimum)")
    print(f"R_end  median {gm.R_end_mOhm.median():8.2f} mOhm")
    print(f"RMSE   median {gm.rmse.median():8.3f} K")
    print(f"noise floor median {gm.sigma.median():.4f} K -> "
          f"adequacy ratio {gm.rmse.median()/gm.sigma.median():.1f}")
    cc = gm.dropna(subset=['cap','R0_mOhm'])
    if len(cc) > 4:
        print(f"\ncorr(capacity, R0)    = {np.corrcoef(cc.cap, cc.R0_mOhm)[0,1]:+.3f}")
        print(f"corr(capacity, R_end) = {np.corrcoef(cc.cap, cc.R_end_mOhm)[0,1]:+.3f}")
    # recovered source shape at the median coefficients
    cs = [gm[f'c{k_+1}'].median() for k_ in range(ORDER)]
    xs = np.linspace(0, 1, 11)
    mult = np.ones_like(xs)
    for k_, ck in enumerate(cs, start=1):
        mult = mult + ck*xs**k_
    print("\nrecovered heat-generation multiplier vs depth of discharge:")
    print("   x   ", "  ".join(f"{v:.1f}" for v in xs))
    print("  R/R0 ", "  ".join(f"{v:.2f}" for v in mult))
    if ORDER == 2 and cs[1] != 0:
        xmin = -cs[0]/(2*cs[1])
        if 0 < xmin < 1:
            print(f"  minimum at x = {xmin:.3f}, multiplier {1+cs[0]*xmin+cs[1]*xmin**2:.3f}")
    dm.to_csv("nasa_final_fits.csv", index=False)
    print("\nsaved nasa_final_fits.csv")

refitting 168 cycles at h = 24.81 W/m2K (profile likelihood), order 2
  previous run used h = 29.565 from truncated cooling branches



usable 168/168 (100.0 %), railed 0

R0     median   152.99 mOhm  IQR/median 0.148
c1     median   -1.352  IQR [-1.444, -1.277]  IQR/|median| 0.124
c2     median    2.192  IQR [2.107, 2.252]  IQR/|median| 0.066
x_min  median    0.318  IQR [0.304, 0.324]   (location of the DCIR minimum)
R_end  median   280.39 mOhm
RMSE   median    0.228 K
noise floor median 0.0063 K -> adequacy ratio 36.4

corr(capacity, R0)    = -0.908
corr(capacity, R_end) = -0.981

recovered heat-generation multiplier vs depth of discharge:
   x    0.0  0.1  0.2  0.3  0.4  0.5  0.6  0.7  0.8  0.9  1.0
  R/R0  1.00  0.89  0.82  0.79  0.81  0.87  0.98  1.13  1.32  1.56  1.84
  minimum at x = 0.308, multiplier 0.792

saved nasa_final_fits.csv


In [22]:
# ===== CELL 7n : IS THE REMAINING STRUCTURE CONVECTIVE? ====================
# The residual sits ~12x the measured noise floor even at the profile h. One
# candidate costs NO extra parameter. Natural convection is not constant: for a
# vertical cylinder Nu ~ Ra^0.25 and Ra ~ dT, so
#       h_conv(dT) = h0 * (dT / dT_ref)^0.25
# Across a 14 K rise that is a ~1.9x swing in h, so a CONSTANT h must be wrong
# at both ends of the cycle. Radiation adds ~5.5 W/m2K at 300 K but is nearly
# linear in dT, so it stays absorbed in h0.
# Same parameter count as before -> a clean structural test, not extra freedom.

DT_REF, DT_MIN = 10.0, 0.25       # K; DT_MIN avoids the singularity at dT -> 0

def theta_exp_hvar(t, q, h0, theta0, L_, rho_cp, dt_ref=DT_REF, dt_min=DT_MIN):
    """Same exponential integrator, but h re-evaluated each step from local dT."""
    th = np.empty_like(t); th[0] = theta0
    dt = np.diff(t)
    for i in range(dt.size):
        h_i = h0*(max(th[i], dt_min)/dt_ref)**0.25
        tau = rho_cp*L_/(2.0*h_i)
        E = np.exp(-dt[i]/tau)
        th[i+1] = th[i]*E + (q[i]/rho_cp)*tau*(1.0 - E)
    return th

def model_poly_hvar(t, R0, coefs, h0, I, x, T_amb, T0, L_, V_, rho_cp):
    mult = np.ones_like(x)
    for k, ck in enumerate(coefs, start=1):
        mult = mult + ck*x**k
    q = I**2*R0*mult/V_
    return T_amb + theta_exp_hvar(t, q, h0, T0-T_amb, L_, rho_cp)

def fit_poly_hvar(t, y, I, x, T_amb, order, L_, V_, rho_cp):
    lo = [0.020] + [-20.0]*order + [2.0]
    hi = [0.300] + [ 20.0]*order + [200.0]
    p0 = [0.09] + [0.5]*order + [30.0]
    def m(tt, *p):
        return model_poly_hvar(tt, p[0], list(p[1:1+order]), p[1+order],
                               I, x, T_amb, y[0], L_, V_, rho_cp)
    popt, pcov = curve_fit(m, t, y, p0=p0, bounds=(lo, hi), method='trf',
                           loss='soft_l1', f_scale=0.5, max_nfev=6000)
    railed = bool(np.any(np.abs(popt-np.array(lo))/np.maximum(np.abs(lo),1e-9) < 0.01)
                  or np.any(np.abs(popt-np.array(hi))/np.abs(hi) < 0.01))
    return popt, m(t, *popt), railed

# head-to-head on the probe cycle
t0, y0, I0 = probe['t']-probe['t'][0], probe['T'], probe['I']
if t0.size > MAXPTS:
    k = np.linspace(0, t0.size-1, MAXPTS).astype(int); t0, y0, I0 = t0[k], y0[k], I0[k]
x0 = dod(t0, I0)
sig = noise_floor(probe['T'])

pc, sdc, yfc, rc_ = fit_poly(t0, y0, I0, x0, probe['T_amb'], H_PROFILE, ORDER,
                             L_NASA, V_NASA, RHOCP_NASA)
rm_const = float(np.sqrt(np.mean((yfc-y0)**2)))
try:
    pv, yfv, rv_ = fit_poly_hvar(t0, y0, I0, x0, probe['T_amb'], ORDER,
                                 L_NASA, V_NASA, RHOCP_NASA)
    rm_var = float(np.sqrt(np.mean((yfv-y0)**2)))
    print(f"probe cycle {probe['idx']}   noise floor {sig:.4f} K")
    print(f"  constant h = {H_PROFILE:.2f}      RMSE {rm_const:.4f} K   "
          f"ratio {rm_const/sig:5.1f}")
    print(f"  h ~ dT^0.25, h0 = {pv[1+ORDER]:.2f}  RMSE {rm_var:.4f} K   "
          f"ratio {rm_var/sig:5.1f}   railed {rv_}")
    print(f"  R0 {pc[0]*1e3:.2f} -> {pv[0]*1e3:.2f} mOhm ; "
          f"c1 {pc[1]:+.3f} -> {pv[1]:+.3f}", end="")
    if ORDER >= 2: print(f" ; c2 {pc[2]:+.3f} -> {pv[2]:+.3f}", end="")
    print()
    print()
    if rv_:
        print("INCONCLUSIVE: the variable-h fit railed; widen the bound that railed.")
    elif rm_var < 0.7*rm_const:
        print("CONVECTIVE SCALING CONFIRMED: temperature-dependent h explains a large")
        print("part of what remained, at no extra parameter cost. Adopt it for the")
        print("stratum-wide fit and re-run Cell 7m with model_poly_hvar.")
    elif rm_var < 0.95*rm_const:
        print("MARGINAL: a real but small improvement. Keep the constant-h model for")
        print("simplicity and record this as a known second-order effect.")
    else:
        print("NOT CONVECTIVE: the dT^0.25 law does not explain the remaining residual.")
        print("Next candidates, in order: ambient/fixture drift (T_amb as a free")
        print("parameter, identifiability checked first), then current-measurement")
        print("scale error, then record segmentation. Do not add polynomial order -")
        print("Cell 7h already showed order 3 is unsupported.")
except Exception as e:
    print("variable-h fit failed:", type(e).__name__, e)

probe cycle 1   noise floor 0.0120 K
  constant h = 24.81      RMSE 0.1297 K   ratio  10.8
  h ~ dT^0.25, h0 = 24.52  RMSE 0.1327 K   ratio  11.0   railed False
  R0 144.35 -> 136.95 mOhm ; c1 -1.305 -> -1.469 ; c2 +1.934 -> +2.285

NOT CONVECTIVE: the dT^0.25 law does not explain the remaining residual.
Next candidates, in order: ambient/fixture drift (T_amb as a free
parameter, identifiability checked first), then current-measurement
scale error, then record segmentation. Do not add polynomial order -
Cell 7h already showed order 3 is unsupported.


In [23]:
# ===== CELL 7o : IS THE NOISE FLOOR TRUSTWORTHY? ===========================
# The stratum-median floor came out at 0.0063 K. Six millikelvin is below any
# plausible T-type thermocouple resolution, and this dataset was produced by a
# notebook named "...-dataset-cleaning". If the series was filtered or resampled,
# the second-difference estimator under-reports sigma and the adequacy ratio is
# inflated by the pre-processing rather than by missing physics.
#
# Exact test. For white noise y,  d2[i] = y[i] - 2y[i+1] + y[i+2]  has
#   var(d2) = 6 sigma^2   and   lag-1 autocorr(d2) = -4/6 = -0.6667 exactly.
# Verified numerically, three distinct regimes:
#   ac1(d2) ~ -0.67  white measurement noise dominates -> sigma_hat is honest
#                    (checked at sigma = 0.10, 0.02, 0.005 K: -0.670, -0.610, -0.689)
#   ac1(d2) ~ -0.50  boxcar-filtered -> sigma_hat under-reports 3.7x (3-pt) to 6.2x (5-pt)
#   ac1(d2) ~  0.00  linearly interpolated -> sigma_hat under-reports ~2.8x
#   ac1(d2) >  0     SIGNAL-CURVATURE DOMINATED: essentially no white noise remains,
#                    so sigma_hat is measuring the signal's own curvature over three
#                    samples, not the instrument. Pure noiseless signal gives +0.99.
# The last regime is the one that voids the adequacy ratio completely.

def floor_diagnostics(y):
    d2 = y[2:] - 2*y[1:-1] + y[:-2]
    sig = float(np.std(d2)/np.sqrt(6.0))
    ac  = float(np.sum(d2[:-1]*d2[1:])/np.sum(d2**2))
    return sig, ac

rows = []
for c in sel[:40]:
    s, a = floor_diagnostics(c['T'])
    rows.append((c['idx'], s, a, np.min(np.abs(np.diff(np.unique(c['T'])))) if
                 np.unique(c['T']).size > 1 else np.nan))
dg = pd.DataFrame(rows, columns=["idx", "sigma", "ac1_d2", "quantum"])
print(f"diagnostics over {len(dg)} cycles")
print(f"  sigma_hat  median {dg.sigma.median():.5f} K")
print(f"  ac1(d2)    median {dg.ac1_d2.median():+.4f}   (white-noise reference -0.6667)")
print(f"  smallest observed temperature step, median {dg.quantum.median():.5f} K")
print()
ac_med = dg.ac1_d2.median()
if abs(ac_med + 0.6667) < 0.08:
    print("NOISE FLOOR TRUSTWORTHY: white noise dominates, so sigma_hat is a fair")
    print("estimate and the adequacy ratio means what it says.")
elif ac_med > -0.08:
    print("NO DETECTABLE MEASUREMENT NOISE: ac1(d2) is at or above zero, so the second")
    print("difference is dominated by the signal's own curvature rather than by noise.")
    print("sigma_hat is therefore not an instrument floor at all and the adequacy ratio")
    print("is void - not merely inflated. This series has been filtered or resampled")
    print("until the white component is gone.")
    print()
    print("What to do: (1) quote the RELATIVE figure of merit below, with the")
    print("pre-processing caveat; (2) if the residual structure matters for a claim,")
    print("attach the .mat mirror (ckskaggle/li-ion-battery-dataset-from-nasa-pcoe),")
    print("which is closer to the original NASA files - Cell 6 handles that layout")
    print("automatically - and re-run this diagnostic on it before drawing any")
    print("conclusion about missing physics.")
    print()
    print(f"  relative figure of merit: RMSE {gm.rmse.median():.3f} K over a rise of "
          f"{gm.rise.median():.2f} K = {100*gm.rmse.median()/gm.rise.median():.2f} %")
else:
    print("NOISE FLOOR NOT TRUSTWORTHY: ac1(d2) departs from -0.6667, so this series")
    print("has been filtered or resampled. sigma_hat UNDER-estimates the true")
    print("measurement noise, which INFLATES every adequacy ratio computed from it.")
    print("Consequence: do not read 'N x the noise floor' as evidence of missing")
    print("physics on this dataset. Use the relative figure of merit instead -")
    print("RMSE divided by the temperature rise - and state the pre-processing")
    print("caveat wherever the number is quoted.")
    print()
    print(f"  relative figure of merit: RMSE {gm.rmse.median():.3f} K over a rise of "
          f"{gm.rise.median():.2f} K = {100*gm.rmse.median()/gm.rise.median():.2f} %")

diagnostics over 40 cycles
  sigma_hat  median 0.01243 K
  ac1(d2)    median +0.3190   (white-noise reference -0.6667)
  smallest observed temperature step, median 0.00135 K

NO DETECTABLE MEASUREMENT NOISE: ac1(d2) is at or above zero, so the second
difference is dominated by the signal's own curvature rather than by noise.
sigma_hat is therefore not an instrument floor at all and the adequacy ratio
is void - not merely inflated. This series has been filtered or resampled
until the white component is gone.

What to do: (1) quote the RELATIVE figure of merit below, with the
pre-processing caveat; (2) if the residual structure matters for a claim,
attach the .mat mirror (ckskaggle/li-ion-battery-dataset-from-nasa-pcoe),
which is closer to the original NASA files - Cell 6 handles that layout
automatically - and re-run this diagnostic on it before drawing any
conclusion about missing physics.

  relative figure of merit: RMSE 0.228 K over a rise of 15.87 K = 1.44 %


### 4.11 The noise-floor question settled — and why it closes Part 3 rather than reopening it

The diagnostic returned **ac1(d2) = +0.3190** against a white-noise reference of −0.6667, with a smallest observed temperature step of **0.00135 K**.

A *positive* value is a regime my original reference table did not contain, and it is the decisive one. Checked numerically on a realistic 197-sample rise: white noise at σ = 0.10, 0.02 and 0.005 K returns −0.670, −0.610, −0.689; a boxcar filter returns ≈ −0.50; linear interpolation returns ≈ 0.00; and a **pure noiseless signal returns +0.99**. Positive autocorrelation therefore means the second difference is dominated by the signal's own smooth curvature, with essentially no white measurement component left. σ_hat = 0.0124 K is measuring curvature over three samples, not an instrument.

So the adequacy ratio is not merely inflated — it is **void**. The "36× the noise floor" figure carries no information about missing physics, and neither would any smaller number computed the same way. Supporting evidence: a quantum of 0.00135 K is neither a decimal fraction nor a power of two, which is characteristic of interpolation or resampling rather than a raw ADC step. This dataset came from a cleaning notebook and the diagnostic detected exactly that.

**The honest figure of merit** is therefore relative: **RMSE 0.228 K over a 15.87 K rise = 1.44 %**, with the pre-processing caveat stated wherever it is quoted.

**Why this closes Part 3.** Chasing the remaining residual on this dataset cannot distinguish model error from pre-processing artifact — the reference against which "adequate" would be judged has been removed by the curation. Two consequences, both clean:

1. The classical baseline stands as delivered: 1.44 % relative on 168 of 168 cycles, a physically sensible non-monotonic source recovered from one surface channel, h from pooled profile likelihood, and **1.14× agreement with an independent instrument**. The EIS check is what makes the absolute scale credible, and it does not depend on the noise floor at all.
2. If the residual structure ever matters for a claim, the route is raw data, not more parameters: attach the `.mat` mirror (`ckskaggle/li-ion-battery-dataset-from-nasa-pcoe`), which sits closer to the original NASA files. Cell 6 handles that layout automatically, and Cell 7o should be re-run on it *before* any conclusion about missing physics is drawn. That is a twenty-minute check, not a work package.

**Method lesson, logged.** This is the fourth metric in Part 3 that a control or a self-test invalidated: sign-change counting, autocorrelation thresholds calibrated on white-noise mocks, naive R₀ ∝ h scaling, and now the noise floor itself. In every case the defect was the same shape — a criterion calibrated on synthetic data cleaner or simpler than the real thing. The tests that caught them were cheap; the conclusions they prevented would have been expensive. Build the control before trusting the metric.

### 4.10 Part 3 closed — final results, external validation, and the D-series settled

**Final fit at the profile-likelihood h (Cell 7m), B0005 @ 24 °C, 168 cycles.**

| Quantity | h = 29.57 (truncated) | **h = 24.81 (profile)** |
|---|---|---|
| usable | 159 / 168 (94.6 %) | **168 / 168 (100 %)**, 0 railed |
| RMSE median | 0.260 K | **0.228 K** |
| R₀ median | 157.5 mΩ | **153.0 mΩ** |
| c₁, c₂ | −1.255, +2.276 | **−1.352, +2.192** |
| R_end median | 306.8 mΩ | **280.4 mΩ** |
| corr(capacity, R₀) | −0.906 | **−0.908** |
| corr(capacity, R_end) | −0.986 | **−0.981** |
| source minimum | x = 0.276, 0.827× | **x = 0.308, 0.792×** |
| rise to cutoff | 2.02× | **1.84×** |

Every cycle now converges inside the gates, and the recovered shape barely moved under a 16 % change in h — minimum at 0.28–0.31 depth of discharge, rise 1.8–2.0× to cutoff. **The shape is robust; that claim is now tested rather than asserted.**

**A prediction of mine, wrong, logged.** Last turn I projected R₀ ≈ 132 mΩ at h = 24.81 by scaling R₀ ∝ h from the steady-state relation. The refit returned **153 mΩ**, a 3 % move rather than 16 %, because the source coefficients re-absorbed most of the change (c₁ −1.255 → −1.352, c₂ +2.276 → +2.192). Naive single-parameter scaling does not describe a multi-parameter refit — exactly the h-versus-shape coupling Cell 7h had already flagged at cond ≈ 1.8 × 10⁴ for "order 2 + h free". I should have read my own conditioning number before projecting. The practical consequence is favourable: **absolute R₀ is far less sensitive to the h estimate than feared**, which strengthens rather than weakens the result.

**Convective scaling refuted at zero parameter cost (Cell 7n).** Constant h at 0.1297 K versus h ∼ ΔT^0.25 at 0.1327 K on the probe cycle — no improvement, no railing. The ΔT^0.25 law does not explain the residual. Clean negative result on a test that cost nothing, and the negative control on constant-h mock data behaved correctly beforehand.

**The noise-floor ratio is a pre-processing artifact, not physics (Cell 7o).** The stratum-median floor came out at 0.0063 K. Six millikelvin is below any plausible T-type thermocouple resolution, and this dataset was produced by a notebook named "…-dataset-cleaning". The exact test: for white noise the second-difference series has lag-1 autocorrelation of **−4/6 = −0.6667**; a 3-point boxcar gives −0.500 and under-reports σ by 3.7×, a 5-point boxcar −0.501 and 6.2×, linear interpolation 0.000 and 2.8×. So the statistic detects filtering against a known reference. **Wherever it flags smoothing, "36× the noise floor" says nothing about missing physics** — use the relative figure of merit instead: **0.228 K over a 14.3 K rise, 1.6 %**.

**External validation — the result worth quoting (Cell 8).** Within the same battery and ambient, 278 impedance records:

| Source | Value |
|---|---|
| EIS R_e median | 55.9 mΩ |
| EIS R_ct median | 77.5 mΩ |
| **EIS R_e + R_ct** | **134.3 mΩ** |
| **thermal R₀ (start of discharge)** | **153.0 mΩ** |
| ratio | **1.14×** |

Two entirely different instruments — electrochemical impedance spectroscopy and a thermal inverse reconstruction from a surface temperature trace — agree to **14 %** on the same cells. The thermal figure carries an assumed ρc_p **(b)** and an h from pooled profile likelihood, so this agreement corroborates the whole absolute chain, not just the fit. It is corroboration and not proof: R_e + R_ct is measured at a specific frequency and bias while R₀ is the start-of-discharge effective resistance including entropic contributions, so exact equality was never expected. Separately, R_end / (R_e+R_ct) = 2.09×, consistent with the recovered shape.

**D-series, final.**

| # | Prediction | Outcome | Verdict |
|---|---|---|---|
| D1 | converge ≥ 90 % | 100 % at the profile h | HIT |
| D2 | median R₀ 40–150 mΩ | **153.0 mΩ** — 2 % above the band | **MISS** (twice: the original band, and my scaling projection of 132) |
| D3 | corr(capacity, R₀) < −0.5 | −0.908 | HIT |
| D4 | h more scattered than R₀ | held on the first pass | HIT |
| D5 | thermal / EIS 1.5–5× | **1.14×** | **MISS — favourable** |

**3 hits, 2 misses.** D5's miss is the interesting one: I predicted the thermal route would over-read by 1.5–5× because it "integrates the whole discharge". That reasoning was wrong — the polynomial model returns R₀ *at x = 0*, which is the correct comparator against EIS taken near full charge. The prediction was mis-specified, not the measurement. Better agreement than predicted is a miss on the ledger and a win for the method.

**Part 3 is closed.** Delivered: a CRLB framework that gated every model choice before fitting; an identifiability result showing {R_int, h, ρc_p} is rank-deficient and that source shape is supportable only to order 2 with h fixed; a working inverse on real Li-ion data recovering a physically sensible non-monotonic heat-generation profile from one surface channel; h from pooled profile likelihood; and 14 % agreement with an independent instrument. Cumulative battery ledger: **B1–B9 all pass, C-series 3 hits / 1 partial / 1 miss, D-series 3 hits / 2 misses.**

**The bar for Part 4.** A PINN reaching 0.23 K on this stratum has matched a three-parameter `curve_fit` and earned nothing. Its case must be made on what this formulation cannot reach — the spatial field, the PCM melt front, parameters that need the full PDE — and never on trajectory RMSE.

### 4.9 h identified, R₀ made quotable, and what the residual still holds

**Profile likelihood (Cell 7l) worked on the real stratum.** Over 28 of the 168 cycles the total residual falls from 0.530 K at h = 8 to a minimum of **0.2296 K at h = 26**, rising again to 0.949 K at h = 60. Parabolic refinement gives **h = 24.81 W m⁻²K⁻¹**, with profile depth (max−min)/min = **3.13** — a well-defined minimum, not a flat curve.

Compare the stage-1 cooling-branch estimate of 29.57 W m⁻²K⁻¹: **19 % high**, in exactly the direction predicted for a decay observed over 0.28 τ. The prediction that truncation would bias τ short and h high is confirmed, and the pooled estimate replaces it.

**Consequence for R₀, and for D2.** At steady state ΔT = q‴L/2h, so R₀ ∝ h:

| h source | h (W m⁻²K⁻¹) | implied R₀ (mΩ) | inside predicted 40–150? |
|---|---|---|---|
| cooling branch, truncated | 29.57 | 157.5 | no |
| profile likelihood, minimum | 26.00 | 138.5 | yes |
| **profile likelihood, refined** | **24.81** | **132.2** | **yes** |

D2 was scored a miss at 157.5 mΩ and attributed to the h bias at the time. Removing the bias moves R₀ inside the band. The miss stands as recorded — the prediction was scored against the number actually produced — but the attribution was correct, and Cell 7m now refits the stratum at the profile h to produce the value that may be quoted.

**Adequacy against the measured noise floor (Cell 7k).** The second-difference estimator returns σ ≈ **0.0120 K** on the probe cycle — NASA's temperature channel is finely resolved. Against it: constant-q sat at ratio **64.4**, the order-2 source at **12.05**. So the source-shape fix removed roughly five-sixths of the excess, and what remains is real structure rather than instrument limit. Worth stating plainly: 0.145 K on a 14 K rise is a 1 % relative error, which is a usable engineering result even with structure left in it.

**One candidate tested at zero parameter cost (Cell 7n).** Natural convection is not constant — for a vertical cylinder Nu ∼ Ra^0.25 and Ra ∝ ΔT, so h ∼ ΔT^0.25, which is a ~1.9× swing across a 14 K rise. Replacing constant h with h₀(ΔT/10)^0.25 keeps the **same parameter count**, so it is a structural test rather than added freedom. Radiation contributes ≈ 5.5 W m⁻²K⁻¹ at 300 K but is nearly linear in ΔT and stays absorbed in h₀. Negative control on constant-h mock data: RMSE 0.0563 → 0.0582 K, correctly reported as no improvement.

**Part 3 status.** The classical baseline is complete and defensible on real Li-ion data: a well-conditioned identifiable parameter set, a physically sensible non-monotonic source recovered from a surface trace, h from pooled profile likelihood, R₀ quotable, ageing correlation −0.906 to −0.986, and every model choice gated by a CRLB computed before the fit. This is the bar Part 4's PINN has to clear. A PINN that reaches 0.26 K on this stratum has matched a `curve_fit` with three parameters and has earned nothing; the case for it has to be made on the spatial field, on the melt front, or on parameters this formulation cannot reach — not on trajectory RMSE.

### 4.8 The adequacy metric corrected, and h recovered by profile likelihood

**Cell 7j's verdict was produced by a mis-calibrated threshold — mine.** On the real probe cycle the order-2 source cut RMSE from 0.773 K to **0.145 K** (5.3×) and the decile spread from 1.85 K to 0.44 K (4.2×), and changed the residual's character from one smooth arc to an irregular pattern. The cell nonetheless reported "structure persists at close to its original strength", because it judged on lag-1 autocorrelation against a reference of ~0.05 taken from a mock with i.i.d. Gaussian noise.

Autocorrelation is **scale-free**: it describes the shape of a residual, not its size. Real thermocouple data carries *correlated* noise — junction thermal mass, ADC filtering, quantisation — so at ~19 s sampling no real residual can reach 0.05 however small it becomes. The reference was unreachable by construction, which makes the test unable to ever return "cleared" on real data. Fourth metric defect in this family, and the same root cause each time: a criterion calibrated on synthetic data that is cleaner than the real thing.

**The scale-aware replacement (Cell 7k).** Estimate the sensor noise floor from the data itself via the second-difference identity — for a smooth signal plus white noise of standard deviation σ, the second difference has variance 6σ² — then judge the residual against it. Signal curvature inflates the estimate, so it is an upper bound on σ and the resulting adequacy ratio is conservative in the safe direction. Verified on the U-shaped mock: the correctly specified model returns **ratio 1.00** (at the noise floor) while the constant-q model returns **21.85**. That is a discriminator with real dynamic range, unlike a scale-free one.

**h recovered without new data (Cell 7l).** h is a property of the fixture — chamber airflow, mounting, surface finish — so it is common to every cycle in a stratum while R₀ and the source shape age. No single cycle carries a cooling branch long enough to pin τ, but 168 cycles constrain a *shared* h jointly. For each trial h the source coefficients are refitted on every cycle and the total residual accumulated; the resulting curve is a profile likelihood. A clear minimum means h is identifiable; a flat curve means it is not, and the cell says so rather than returning a number.

Validated on the mock at a true h = 12.0 W m⁻²K⁻¹: minimum located at **12.0**, parabolic refinement **11.65**, profile depth (max−min)/min = **37×** — no ambiguity. Against a truncation-biased cooling-branch estimate this is strictly better information, because it uses the whole discharge of every cycle rather than 250 s of tail from each.

**Why this beats both alternatives that were on the table.** Choosing a stratum with longer rest records loses the B0005 cycle series and its paired EIS records, and every NASA discharge record shares the same truncated structure, so the gain is uncertain. Concatenating each discharge with the following record's leading rest is sound but depends on whether the dataset preserves that rest, and it needs new ingestion code. The profile likelihood needs neither: it runs on data already loaded, and its flatness test states honestly when the answer is not there. Concatenation remains the fallback if the profile comes back flat.

### 4.7 Working inverse on real Li-ion data — results and D-series scored

**Stratum B0005 @ 24 °C, 168 discharge cycles, order-2 source, h = 29.565 W m⁻²K⁻¹ fixed from stage 1.**

| Quantity | Value |
|---|---|
| usable cycles | **159 / 168 (94.6 %)**, 0 railed, 0 solver failures |
| RMSE median | **0.260 K** (gate 0.30 K; the monotonic model gave ~0.70 K) |
| R₀ median | 157.5 mΩ, IQR/median 0.161 |
| c₁, c₂ median | −1.255, +2.276 |
| R_end median | 306.8 mΩ |
| corr(capacity, R₀) | **−0.906** |
| corr(capacity, R_end) | **−0.986** |

**The recovered shape.** R(x)/R₀ = 1 − 1.255x + 2.276x² has its minimum at **x = 0.276**, dips to **0.827** there, and reaches **2.02×** at cutoff: 157 mΩ → 130 mΩ at 28 % depth of discharge → 318 mΩ at full discharge. That is the textbook LCO DCIR profile — minimum at mid-to-high SOC, steep climb toward cutoff — recovered from a surface temperature trace and a current record, with no resistance measurement and no assumed entropic coefficient anywhere in the chain.

**Predictions scored (D-series, second pass).**

| # | Prediction | Outcome | Verdict |
|---|---|---|---|
| D1 | converge ≥ 90 % of cycles | 94.6 % under proper gates | HIT |
| D2 | median R_int in 40–150 mΩ **(b)** | **157.5 mΩ**, just above the band | **MISS — attributed** (see scaling caveat) |
| D3 | corr(capacity, R_int) < −0.5 | −0.906 | HIT |
| D4 | h more scattered than R_int | held from the earlier run; h is now fixed, so not re-testable in this configuration | HIT (first pass) |
| D5 | thermal R vs EIS 1.5–5× | Cell 8, pending | open |

**D2 scaling caveat, stated plainly.** At steady state ΔT = q‴L/2h, so q‴ ∝ h and R₀ inherits h linearly. The stage-1 h came from cooling branches spanning ≈ 0.28 τ, below the span/τ ≥ 0.5 gate, so it is truncation-biased — most likely high, which would push R₀ high in exactly the observed direction. At h = 20 W m⁻²K⁻¹ the same data gives R₀ = 106 mΩ, inside the predicted band. R₀ also scales inversely with the assumed ρc_p, inherited from Part 2 **(b)** and not measured for these cells. **Absolute resistances are provisional; the shape coefficients and the ageing trends are not affected**, because both are ratios within a cycle. This is the trajectory-versus-parameter tier distinction again, now controlling how the result may be quoted.

**A metric defect of mine, logged.** Cell 7j originally judged "is the structure gone?" by counting sign changes in the binned residual. That is backwards: white noise crosses zero often, while a structured error crosses rarely but with large excursions. On the U-shaped mock — where the model is exactly correct — the metric reported 4 sign changes and declared "structure persists". The verdict now rests on lag-1 autocorrelation and the structure ratio std(decile means)/RMSE, with reference values from both extremes: constant-q on real data gave autocorr +0.99 and ratio 0.83; a correctly specified model on mock data gave +0.05 and 0.37. Third harness defect caught by a control rather than by inspection, which is the argument for keeping the controls.

**Next, in order.** (1) Run Cell 7j on the real stratum — if the autocorrelation collapses from +0.99 toward zero, the mechanism was explained rather than absorbed, and Part 3 is closed. (2) Run Cell 8 for the EIS cross-check, now reading from the polynomial fits. (3) Fix h properly: either select a stratum whose records carry a longer rest, or concatenate each discharge with the following record's leading rest, which converts R₀ from provisional to quotable.

### 4.6 The mechanism, identified — a non-monotonic heat source

Cell 7f discriminated the three candidates cleanly.

| Evidence | Reading |
|---|---|
| Binned residual: **+0.42, +1.17, +0.76, +0.10, −0.29, −0.60, −0.68, −0.28, +0.75, +0.96** | data hotter EARLY and LATE, cooler MID — one full oscillation, 2 sign changes |
| β improved load 0.726 → 0.569 K but worsened rest 1.124 → 1.408 K | it bought curvature in one place by paying for it in another: mis-specification, not a parameter error |
| Two-node probe: 0.725 K vs best single-node 0.696 K, R_cs railed at 20 K/W (k = 0.061 W m⁻¹K⁻¹, below physical) | **internal gradient refuted a second time**, now on real data |

A monotonic ramp cannot bend twice. The signature is what a **U-shaped** source leaves under a monotonic model, and a U shape is exactly what LCO physics predicts **(b)**: DCIR is high at full charge, falls to a mid-SOC minimum, then climbs steeply toward cutoff, while the entropic term changes sign with SOC. Neither is assumed numerically anywhere — the shape is *inferred from the temperature trace*, which is precisely the inverse problem this project exists to solve. The real data walked us into the thesis contribution.

**Model.** R(x) = R₀(1 + c₁x + c₂x² + … + c_nxⁿ), x = coulomb-counted depth of discharge.

**Identifiability, order by order (Cell 7h, h fixed):**

| Model | scaled cond(F) | worst relative CRLB sd |
|---|---|---|
| order 1 | 8.79×10² | 1.02 % (c₁) |
| **order 2** | **5.98×10⁴** | **5.78 % (c₁)** |
| order 3 | 4.05×10⁶ | 46.5 % (c₂) |
| order 4 | 2.30×10⁸ | 336 % (c₃) |
| any order **+ h free** | ≥ 8.79×10⁶ | ≥ 103 % (h) |

**Order 2 is the ceiling**, and h must come from elsewhere in every case — the same conclusion Cell 7b reached for β, now confirmed across four model orders. A quadratic is exactly enough to represent one bend, which is exactly what the residual shows.

**Mock validation.** Ten cycles generated with a genuinely U-shaped source (c₁ = −2.0, c₂ = +4.0, minimum at x = 0.25, 3× heat at cutoff), h = 12.0, R₀ ageing, and a 19-point rest tail matching the real records. Recovered: **c₁ = −1.999, c₂ = +4.000**, RMSE median 0.050 K, 10/10 usable, corr(capacity, R₀) = −0.986. For comparison the monotonic model on the same data sits near 0.70 K, the same figure the real data returns.

**Caveat carried forward.** In the mock the rest tail was too short to pass the span/τ gate, so the fit ran with h free and still recovered the source shape — because the mock is structure-matched and lightly noised. The CRLB says h free carries >100 % uncertainty, so on real data the source shape may be recoverable while h is not. Report the shape; report h only if a cooling branch passes the span gate.

### 4.5 Real-data findings, D-series rescored, and a hypothesis of mine refuted

**What the 26 Jul run established.**

| Observation | Reading |
|---|---|
| 168/168 fail **only** the RMSE gate | not truncation, not weak signal, not bounds |
| lag-1 autocorrelation **+0.9947**, RMSE 0.773 K on a ~14 K rise | residual is 5.5 % structure, not noise; the gate must not be relaxed |
| residual thirds **+0.743, −0.397, +0.358 K** | oscillatory, not monotonic |
| β enrichment: still 168/168 fail | **the source-ramp hypothesis is refuted** |
| cooling branches: 97 of 168 found, but span ≈ 250 s against τ ≈ 888 s | span/τ ≈ 0.28 — the h = 27.9 W m⁻²K⁻¹ estimate is **truncation-biased** and must not be quoted; tight IQR does not mean accurate |
| stage 2 with h fixed: still 168/168 fail | fixing h does not rescue it either |

Cell 7d now gates on span/τ ≥ 0.5 and labels the truncated estimate explicitly rather than passing it downstream.

**A hypothesis of mine, registered and then refuted.** I proposed that the misfit came from the internal radial gradient: Part 2 assumed k = 2.5 W m⁻¹K⁻¹ (effective, flagged moderate confidence), whereas the radial conductivity of a wound 18650 is ~0.2–0.5 W m⁻¹K⁻¹ **(b)**, which with the stage-1 h puts Bi = hr/k in 0.5–1.3 rather than ≪ 1. On that hypothesis the surface sensor reads a lagged, attenuated core and a single node cannot suffice.

The controls refuted it. Two mocks were built: one from single-node physics, one from a genuine two-node system with physically plausible parameters (f = 0.75, R_cs = 3.0 K/W, h = 25 W m⁻²K⁻¹, R₀ = 85 mΩ, β = 0.8). On the **two-node** data, the single-node model with a ramped source fit to **0.059 K**. The two structures are therefore observationally equivalent at 18650 scale and these rates — a second node adds no signature the source ramp cannot absorb.

That closes the argument: if the real misfit were an internal-gradient effect, the β-model would have absorbed it to ~0.06 K. It did not; it failed at 0.77 K on every cycle. **The internal gradient is not the missing physics.** Logged as a miss.

Two further methodological notes, both worth more than the hypothesis was. First, the two-node probe as originally written declared "structure refuted" on single-node data, because it compared against the *constant-q* fit rather than the best single-node fit, and because it railed R₀ at an unphysically wide 1 Ω bound. A five-parameter probe with loose bounds can mimic a source error and cannot testify about structure. Bounds are now physical (R₀ 20–300 mΩ, h 2–100 W m⁻²K⁻¹, R_cs 0.5–20 K/W) and rail detection is mandatory. Second, a variant holding h at the single-node value was tried and is mis-specified — in a two-node system the surface node carries only (1−f)·C_tot, so the same h implies different dynamics — and it is not retained.

**Remaining candidates, and the test that discriminates.** In order of prior plausibility **(c)**:

1. **A genuinely non-monotonic heat source.** The entropic term changes sign with SOC, and at 2 A it is 20–40 % of the ohmic term. A monotonic β-ramp cannot represent a sign change; the oscillatory residual is consistent with exactly this. Note the project rule: dV_oc/dT is measured or the term is disabled, never invented — so the route here is a shape-flexible source (piecewise-linear in DOD), not an assumed coefficient.
2. **Ambient or fixture drift.** T_amb is a per-cycle scalar from metadata, not a logged trace. A chamber drift of ~1 K over 3700 s would produce a slow structured residual.
3. **Record segmentation.** The record may contain rest → discharge → rest, and possibly a partial charge; a segment that violates the model would show up as a phase-localised residual.

Cell 7f discriminates these directly: it reports the residual RMSE **split by load and rest phase** and the mean residual **by decile of elapsed time**. Candidate 1 predicts the misfit concentrates in the load phase with two or more sign changes; candidate 2 predicts a slow drift with comparable error in both phases; candidate 3 predicts a phase-localised jump. Run 7f before any further model change.

**Mock validation of the enriched path (build container).** Twelve cycles generated with $\beta = 1.5$, $h = 11.0$ W m⁻²K⁻¹ and $R_0$ ageing +2 %/cycle, then passed through the full discovery → stratify → diagnose → fit chain. Recovery: β median 2.305 against a true 1.500, h median 17.146 against a true 11.0, 11/12 cycles usable. The two-parameter fitter on the same data fails the RMSE gate on every cycle, reproducing the real-data signature — which is the point of the mock: it demonstrates that the diagnosis, not just the code, is correct.

## 5. The external check that makes this worth doing

The thermally identified $R_\mathrm{int}$ and the EIS-measured $R_e + R_{ct}$ come from entirely different instruments on the same cell. Agreement in *magnitude* would be a real external validation of the inverse; agreement in *trend across ageing* is the stronger and more likely result, since the thermal route inherits the $\rho c_p$ assumption.

Do not expect equality. EIS resistance is measured at a specific frequency and DC bias; the thermal route recovers an effective DC resistance integrated over the discharge. **(c)** A factor-of-two offset with a matching ageing trend would be a good outcome and should be reported as such rather than tuned away.

In [24]:
# ===== CELL 8 : EIS CROSS-CHECK, MATCHED TO THE SAME STRATUM ================
Re = Rct = None
if LAYOUT == "csv" and META_PATH and os.path.exists(META_PATH):
    m2 = pd.read_csv(META_PATH)
    imp = m2[m2["type"].astype(str).str.lower() == "impedance"]
    if BATTERY and "battery_id" in imp.columns:
        imp = imp[imp["battery_id"].astype(str) == BATTERY]
    if AMBIENT is not None and "ambient_temperature" in imp.columns:
        imp = imp[np.isclose(pd.to_numeric(imp["ambient_temperature"],
                                           errors="coerce"), AMBIENT)]
    Re  = pd.to_numeric(imp.get("Re"),  errors="coerce").dropna().to_numpy()
    Rct = pd.to_numeric(imp.get("Rct"), errors="coerce").dropna().to_numpy()
elif LAYOUT == "mat":
    from scipy.io import loadmat
    mm = loadmat(os.path.join(CYCLE_DIR, (BATTERY or "B0005")+".mat"), simplify_cells=True)
    key = [k for k in mm if not k.startswith("__")][0]
    eis = [c["data"] for c in mm[key]["cycle"] if c["type"] == "impedance"]
    Re  = np.array([float(np.real(np.atleast_1d(e["Re"])[0]))  for e in eis])
    Rct = np.array([float(np.real(np.atleast_1d(e["Rct"])[0])) for e in eis])

# prefer the stage-2 results; fall back to the 2-parameter fit if stage 2 was skipped
_src = None
if 'gm' in dir() and len(gm) >= 3:
    _src, _therm, _lab = gm, gm.R0_mOhm.median(), "thermal R0 at profile h"
elif 'gp' in dir() and len(gp) >= 3:
    _src, _therm, _lab = gp, gp.R0_mOhm.median(), "poly R0 (start of discharge)"
elif 'g2' in dir() and len(g2) >= 3:
    _src, _therm, _lab = g2, g2.R0_mOhm.median(), "stage-2 R0 (h fixed)"
elif len(good) >= 3:
    _src, _therm, _lab = good, good.R_mOhm.median(), "2-parameter R_int"

if Re is not None and len(Re) and _src is not None:
    tot, therm = np.median(Re+Rct)*1e3, _therm
    print(f"stratum {BATTERY} @ {AMBIENT} C, {len(Re)} impedance records")
    print(f"  EIS Re      median {np.median(Re)*1e3:8.2f} mOhm")
    print(f"  EIS Rct     median {np.median(Rct)*1e3:8.2f} mOhm")
    print(f"  EIS Re+Rct  median {tot:8.2f} mOhm")
    print(f"  {_lab:18s} median {therm:8.2f} mOhm")
    print(f"  ratio thermal/EIS  {therm/tot:8.2f}x")
    print("\nCompare only within one battery and one ambient. Rct is strongly")
    print("temperature dependent, so a pooled ratio across ambients is meaningless.")
else:
    print("EIS check unavailable: need impedance rows for this stratum and >=3\nusable fits from Cell 7e (preferred) or Cell 7.")

stratum B0005 @ 24.0 C, 278 impedance records
  EIS Re      median    55.85 mOhm
  EIS Rct     median    77.49 mOhm
  EIS Re+Rct  median   134.32 mOhm
  thermal R0 at profile h median   152.99 mOhm
  ratio thermal/EIS      1.14x

Compare only within one battery and one ambient. Rct is strongly
temperature dependent, so a pooled ratio across ambients is meaningless.


## 6. Status and what comes next

**Status.** CRLB and identifiability established and executed; lumped inverse validated against synthetic truth; real-data path written and awaiting its first Kaggle run.

**Next, in order:** (1) run Cells 6–8 on Kaggle, paste outputs; (2) score the C-series predictions and the real-data behaviour honestly — a miss here is more informative than a hit; (3) *only then* the forward PINN as a machinery gate, scored against this CRLB rather than against a plot. The transformer campaign's rule stands: a PINN earns its place by matching or beating the classical bound, not by producing a curve that looks right.

### 4.12 The noise-floor question, settled — negatively, on the raw files

Cell 7o was re-run on the raw `.mat` mirror (`ckskaggle/li-ion-battery-dataset-from-nasa-pcoe`), which sits closer to the original NASA acquisition than the cleaned CSV.

| diagnostic | cleaned CSV | **raw `.mat`** |
|---|---|---|
| σ̂ (2nd-difference) | 0.01243 K | **0.01243 K** |
| ac1(d2) | +0.3190 | **+0.3190** |
| smallest observed step | 0.00135 K | **0.00135 K** |

**Identical to five significant figures.** The smoothing is therefore in the **original NASA files**, not introduced by the cleaning notebook. §4.11 recommended attaching the `.mat` mirror to settle this; that recommendation has now been executed and **refuted**. No white-noise floor is recoverable from this dataset in any form, and no further processing route will produce one.

**Three consequences, stated so they are not re-litigated.**

1. **The adequacy ratio is void on this dataset, permanently.** Not inflated, not correctable — the reference against which "adequate" would be judged does not exist in the data. The honest figure of merit is relative: **RMSE 0.228 K over a 15.87 K rise = 1.44 %**, quoted with the pre-processing caveat.
2. **Identifiability conclusions survive; absolute bounds are conditional.** The Fisher matrix is F = SSᵀ/σ², so cond(F) is **σ-independent** — the rank deficiency at 8.8×10⁹, the order-2 source ceiling, and the >100 % bound with h free all stand exactly as reported. But sd values scale **linearly** with σ, so the quoted CRLB standard deviations (0.116 % on R_int, 7.48 % on c₁, and every other absolute bound) are conditional on an assumed σ = 0.10 K that this dataset cannot confirm. Report them as conditional, not measured.
3. **Any claim that needs a real noise floor needs different data.** Not a different mirror — different instrumentation. That is precisely what the DAQ hardware in §3.1 and the instrumented-cell route in §3.2 provide, and it is now a documented reason for wanting them rather than a preference.

### 4.13 Cross-mirror replication — every classical number identical

Running the full pipeline on the raw `.mat` files reproduced the CSV-derived results exactly:

| quantity | both mirrors |
|---|---|
| R₀ median | 152.99 mΩ |
| c₁, c₂ | −1.352, +2.192 |
| R_end median | 280.39 mΩ |
| RMSE median | 0.228 K |
| h (profile likelihood) | 24.81 W m⁻²K⁻¹ |
| corr(capacity, R₀) | −0.908 |
| thermal / EIS | 1.14× |

Two independently processed distributions of the same underlying measurements, loaded by different code paths (`.mat` struct parsing versus CSV with metadata), give identical results to five significant figures. That is a stronger reproducibility statement than the two-environment CPU check, because it varies the *data pipeline* rather than the compute.

Note also that the `.mat` layout carries no `metadata.csv`, so Cell 6c reported "cannot stratify" — correctly. Stratification is implicit there: `B0005.mat` **is** the B0005 stratum, and the 168 discharge cycles it yields match the 168 selected by metadata on the CSV side.

### 4.14 Shape spread now reported — and the most stable quantity in the analysis

| parameter | median | IQR | IQR / \|median\| |
|---|---|---|---|
| R₀ | 152.99 mΩ | 22.71 | 14.8 % |
| c₁ | −1.352 | 0.167 | 12.4 % |
| c₂ | +2.192 | 0.144 | 6.6 % |
| **x_min** | **0.318** | **[0.304, 0.324]** | **6.3 %** |

The **location of the DCIR minimum is the most stable quantity in the whole analysis** — more stable than either shape coefficient and less than half the variability of the amplitude. Physically that is what one expects if the minimum is set by the electrode chemistry while the amplitude tracks ageing, and it makes x_min the natural chemistry fingerprint: a single number, recovered from a surface temperature trace, that identifies the cell type and stays put as the cell degrades.

Note that median(x_min) = 0.318 differs slightly from the 0.308 computed by evaluating −c₁/2c₂ at the median coefficients. The median of a ratio is not the ratio of medians; the per-cycle value with its IQR is the one to quote.